# 🌴 Date Palm Yield Forecasting — Data Collection Pipeline
## Southern Tunisia Oases: Tozeur, Kébili, Gafsa, Gabès | 2002–2024

**Project**: Integrated Date Palm Production Forecasting & Aquifer Sustainability Monitoring  
**Purpose**: Collect, harmonize, and compile all data sources into a model-ready dataset  

### Pipeline structure

| Phase | Cells | Description |
|-------|-------|-------------|
| **A. Setup** | 1–3 | Install packages, authenticate GEE, define study area |
| **B. Satellite extraction** | 4–10 | MODIS NDVI/LST/ET, ERA5-Land, CHIRPS, GRACE, Sentinel-2 |
| **C. Static data** | 11 | SoilGrids, SRTM DEM |
| **D. ONAGRI integration** | 12–13 | Upload & merge ground truth (production + aquifer) |
| **E. GRACE calibration** | 14 | Downscale TWS using CRDA extraction data |
| **F. Master dataset** | 15–16 | Compile annual flat matrix + monthly sequences |

### Output folder structure
```
Google Drive/
└── DatePalm_Project/
    ├── 01_raw_gee/          ← Individual satellite CSVs
    ├── 02_onagri/           ← Uploaded ONAGRI files
    ├── 03_compiled/         ← Merged master datasets
    └── 04_documentation/    ← Data dictionary, calibration report
```


---
## Phase A — Setup & Configuration
### Cell 1: Install packages and import libraries


In [ ]:
# Install required packages (Colab has most pre-installed)
!pip install -q geemap earthengine-api

import ee
import geemap
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime, timedelta
from google.colab import drive, files
import warnings
warnings.filterwarnings('ignore')

print("✓ All packages imported")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.9 MB/s eta 0:00:00
✓ All packages imported


### Cell 2: Authenticate GEE & mount Google Drive
> **First-time setup**: Uncomment `ee.Authenticate()` and follow the OAuth flow.  
> Replace `'your-gee-project-id'` with your actual GEE cloud project ID.


In [ ]:
# --- GEE Authentication ---
ee.Authenticate()  # Uncomment on first run — opens OAuth window
ee.Initialize(project='bonplan-6c907')  # ← REPLACE THIS

# --- Google Drive ---
drive.mount('/content/drive')

# --- Create project folder structure ---
BASE_DIR = '/content/drive/MyDrive/DatePalm_Project'
DIRS = {
    'raw_gee':   os.path.join(BASE_DIR, '01_raw_gee'),
    'onagri':    os.path.join(BASE_DIR, '02_onagri'),
    'compiled':  os.path.join(BASE_DIR, '03_compiled'),
    'docs':      os.path.join(BASE_DIR, '04_documentation'),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("✓ GEE initialized")
print(f"✓ Project folder: {BASE_DIR}")
for key, path in DIRS.items():
    print(f"  ├── {key}: {path}")


Mounted at /content/drive
✓ GEE initialized
✓ Project folder: /content/drive/MyDrive/DatePalm_Project
  ├── raw_gee: /content/drive/MyDrive/DatePalm_Project/01_raw_gee
  ├── onagri: /content/drive/MyDrive/DatePalm_Project/02_onagri
  ├── compiled: /content/drive/MyDrive/DatePalm_Project/03_compiled
  ├── docs: /content/drive/MyDrive/DatePalm_Project/04_documentation


### Cell 3: Define study area — governorate boundaries & time range

We use **FAO GAUL Level 1** administrative boundaries for Tunisia.  
The 4 target governorates cover the main oasis zones where >95%% of Tunisian dates are produced.


In [ ]:
# --- Administrative boundaries ---
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

# GAUL may use French or ASCII spellings — try both
target_names = ['Tozeur', 'Kebili', 'Kébili', 'Gafsa', 'Gabes', 'Gabès']

study_area = gaul.filter(
    ee.Filter.And(
        ee.Filter.eq('ADM0_NAME', 'Tunisia'),
        ee.Filter.inList('ADM1_NAME', target_names)
    )
)

found_names = study_area.aggregate_array('ADM1_NAME').getInfo()
print(f"✓ Found {len(found_names)} governorate(s): {found_names}")

if len(found_names) < 4:
    print("⚠ Missing governorates. Printing all Tunisian ADM1 names for debugging:")
    all_tunisia = gaul.filter(ee.Filter.eq('ADM0_NAME', 'Tunisia'))
    print(all_tunisia.aggregate_array('ADM1_NAME').getInfo())

# Build per-governorate geometry dict with standardized names
NAME_STANDARDIZE = {
    'Tozeur': 'Tozeur', 'Kebili': 'Kebili', 'Kébili': 'Kebili',
    'Gafsa': 'Gafsa', 'Gabes': 'Gabes', 'Gabès': 'Gabes'
}

gov_geometries = {}
for name in found_names:
    std_name = NAME_STANDARDIZE.get(name, name)
    gov_geometries[std_name] = study_area.filter(
        ee.Filter.eq('ADM1_NAME', name)
    ).geometry()

# Full region geometry (for GRACE — coarse resolution)
region_geometry = study_area.geometry()

# --- Time parameters ---
START_DATE = '2002-01-01'
END_DATE   = '2024-12-31'
YEARS  = list(range(2002, 2025))
MONTHS = list(range(1, 13))
GOV_NAMES = sorted(gov_geometries.keys())

print(f"✓ Governorates: {GOV_NAMES}")
print(f"✓ Period: 2002–2024 ({len(YEARS)} years × {len(MONTHS)} months)")
print(f"✓ Expected monthly records per source: {len(GOV_NAMES)} × {len(YEARS)} × {len(MONTHS)} = {len(GOV_NAMES)*len(YEARS)*len(MONTHS)}")


✓ Found 4 governorate(s): ['Gabes', 'Gafsa', 'Kebili', 'Tozeur']
✓ Governorates: ['Gabes', 'Gafsa', 'Kebili', 'Tozeur']
✓ Period: 2002–2024 (23 years × 12 months)
✓ Expected monthly records per source: 4 × 23 × 12 = 1104


### Cell 3b: Reusable extraction helpers

These functions handle the repetitive GEE `reduceRegion` calls with error handling,  
progress logging, and automatic CSV saving.


In [ ]:
def extract_monthly(collection, bands, gov_geometries, scale=1000,
                    description="", save_as=None):
    """
    Extract monthly zonal statistics for one or more bands across all governorates.

    Parameters
    ----------
    collection : ee.ImageCollection — pre-filtered to date range, bands selected
    bands : list of str — band names to extract
    gov_geometries : dict — {name: ee.Geometry}
    scale : int — spatial resolution in meters for reduceRegion
    description : str — printed progress label
    save_as : str or None — filename to save in 01_raw_gee/

    Returns
    -------
    pd.DataFrame with columns: governorate, year, month, {band}_mean, {band}_max, {band}_std
    """
    results = []
    total = len(gov_geometries) * len(YEARS) * len(MONTHS)
    counter = 0

    for gov_name, geom in gov_geometries.items():
        for year in YEARS:
            for month in MONTHS:
                counter += 1
                start = ee.Date.fromYMD(year, month, 1)
                end = start.advance(1, 'month')
                monthly = collection.filterDate(start, end)

                row = {'governorate': gov_name, 'year': year, 'month': month}

                try:
                    n = monthly.size().getInfo()
                    if n == 0:
                        for b in bands:
                            row[f'{b}_mean'] = None
                            row[f'{b}_max'] = None
                            row[f'{b}_std'] = None
                        row['image_count'] = 0
                    else:
                        composite = monthly.reduce(
                            ee.Reducer.mean()
                            .combine(ee.Reducer.max(), sharedInputs=True)
                            .combine(ee.Reducer.stdDev(), sharedInputs=True)
                        )
                        vals = composite.reduceRegion(
                            reducer=ee.Reducer.mean(),
                            geometry=geom, scale=scale, maxPixels=1e9
                        ).getInfo()

                        for b in bands:
                            row[f'{b}_mean'] = vals.get(f'{b}_mean')
                            row[f'{b}_max']  = vals.get(f'{b}_max')
                            row[f'{b}_std']  = vals.get(f'{b}_stdDev')
                        row['image_count'] = n

                except Exception as e:
                    row['error'] = str(e)

                results.append(row)

                if counter % 100 == 0:
                    print(f"  [{description}] {counter}/{total} "
                          f"({counter/total*100:.0f}%) — {gov_name} {year}-{month:02d}")

    df = pd.DataFrame(results)

    if save_as:
        path = os.path.join(DIRS['raw_gee'], save_as)
        df.to_csv(path, index=False)
        print(f"✓ Saved {save_as}: {df.shape[0]} rows × {df.shape[1]} cols")

    return df


def extract_chirps_monthly(gov_geometries, save_as=None):
    """
    Special extractor for CHIRPS: computes monthly total, max daily, and rain day count.
    """
    chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterDate(START_DATE, END_DATE)
    results = []
    total = len(gov_geometries) * len(YEARS) * len(MONTHS)
    counter = 0

    for gov_name, geom in gov_geometries.items():
        for year in YEARS:
            for month in MONTHS:
                counter += 1
                start = ee.Date.fromYMD(year, month, 1)
                end = start.advance(1, 'month')
                monthly = chirps.filterDate(start, end).select('precipitation')

                row = {'governorate': gov_name, 'year': year, 'month': month}

                try:
                    n = monthly.size().getInfo()
                    if n == 0:
                        row.update({'precip_total_mm': None, 'precip_max_daily_mm': None,
                                    'rain_days': None})
                    else:
                        total_img = monthly.sum()
                        max_img = monthly.max()
                        rain_days_img = monthly.map(
                            lambda img: img.gt(0.1).rename('rd')
                        ).sum()

                        for img, key in [(total_img, 'precip_total_mm'),
                                         (max_img, 'precip_max_daily_mm'),
                                         (rain_days_img, 'rain_days')]:
                            band = img.bandNames().getInfo()[0]
                            val = img.reduceRegion(
                                reducer=ee.Reducer.mean(), geometry=geom,
                                scale=5000, maxPixels=1e9
                            ).getInfo().get(band)
                            row[key] = val
                except Exception as e:
                    row['error'] = str(e)

                results.append(row)
                if counter % 100 == 0:
                    print(f"  [CHIRPS] {counter}/{total} ({counter/total*100:.0f}%)")

    df = pd.DataFrame(results)
    if save_as:
        path = os.path.join(DIRS['raw_gee'], save_as)
        df.to_csv(path, index=False)
        print(f"✓ Saved {save_as}: {df.shape[0]} rows × {df.shape[1]} cols")
    return df


print("✓ Helper functions defined")


✓ Helper functions defined


---
## Phase B — Satellite Data Extraction

Each cell below extracts one data source. They are independent and can be run  
in any order (or re-run individually if one fails). Each saves a CSV to `01_raw_gee/`.

**Estimated runtime**: 2–4 hours total for all sources (GEE rate limits apply).  
Recommendation: run Cells 4–6 first (MODIS — critical), then the rest.


### Cell 4: 🟢 MODIS NDVI — Vegetation health (CRITICAL)
**Product**: `MOD13A2.061` (Terra, 1 km, 16-day composites)  
**Band**: `NDVI` (scale factor 0.0001 → dimensionless -1 to 1)  
**Relevance**: Primary proxy for canopy health, phenological stage, and fruit development.  
The pollination window (Mar–Apr) and fruit growth (Jun–Sep) are the critical periods for date palm.


In [ ]:
modis_ndvi = (
    ee.ImageCollection('MODIS/061/MOD13A2')
    .filterDate(START_DATE, END_DATE)
    .select('NDVI')
    .map(lambda img: img.multiply(0.0001).copyProperties(img, ['system:time_start']))
)

df_ndvi = extract_monthly(
    modis_ndvi, bands=['NDVI'], gov_geometries=gov_geometries,
    scale=1000, description='MODIS NDVI', save_as='01_modis_ndvi_monthly.csv'
)

print(f"\nSample:\n{df_ndvi[df_ndvi['governorate']=='Tozeur'].head()}")


  [MODIS NDVI] 100/1104 (9%) — Gabes 2010-04
  [MODIS NDVI] 200/1104 (18%) — Gabes 2018-08
  [MODIS NDVI] 300/1104 (27%) — Gafsa 2003-12
  [MODIS NDVI] 400/1104 (36%) — Gafsa 2012-04
  [MODIS NDVI] 500/1104 (45%) — Gafsa 2020-08
  [MODIS NDVI] 600/1104 (54%) — Kebili 2005-12
  [MODIS NDVI] 700/1104 (63%) — Kebili 2014-04
  [MODIS NDVI] 800/1104 (72%) — Kebili 2022-08


  [MODIS NDVI] 900/1104 (82%) — Tozeur 2007-12
  [MODIS NDVI] 1000/1104 (91%) — Tozeur 2016-04
  [MODIS NDVI] 1100/1104 (100%) — Tozeur 2024-08
✓ Saved 01_modis_ndvi_monthly.csv: 1104 rows × 7 cols

Sample:
    governorate  year  month  NDVI_mean  NDVI_max  NDVI_std  image_count
828      Tozeur  2002      1   0.096424  0.103453  0.007029            2
829      Tozeur  2002      2   0.089284  0.092800  0.003516            2
830      Tozeur  2002      3   0.088454  0.094495  0.006040            2
831      Tozeur  2002      4   0.092102  0.095265  0.003163            2
832      Tozeur  2002      5   0.078114  0.084550  0.006436            2


### Cell 5: 🔴 MODIS LST — Temperature for GDD & heat stress (CRITICAL)
**Product**: `MOD11A2.061` (Terra, 1 km, 8-day composites)  
**Bands**: `LST_Day_1km`, `LST_Night_1km` (raw units: Kelvin × 50)  
**Relevance**: Growing Degree Days require daily temperature. Use **18°C base temperature**  
for date palm (not 10°C). Days >45°C during fruit set cause quality loss in Deglet Nour.


In [ ]:
def lst_to_celsius(img):
    day = img.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('LST_Day_C')
    night = img.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('LST_Night_C')
    return day.addBands(night).copyProperties(img, ['system:time_start'])

modis_lst = (
    ee.ImageCollection('MODIS/061/MOD11A2')
    .filterDate(START_DATE, END_DATE)
    .map(lst_to_celsius)
)

df_lst = extract_monthly(
    modis_lst, bands=['LST_Day_C', 'LST_Night_C'], gov_geometries=gov_geometries,
    scale=1000, description='MODIS LST', save_as='02_modis_lst_monthly.csv'
)

print(f"\nSample:\n{df_lst[df_lst['governorate']=='Kebili'].head()}")


  [MODIS LST] 100/1104 (9%) — Gabes 2010-04
  [MODIS LST] 200/1104 (18%) — Gabes 2018-08
  [MODIS LST] 300/1104 (27%) — Gafsa 2003-12
  [MODIS LST] 400/1104 (36%) — Gafsa 2012-04
  [MODIS LST] 500/1104 (45%) — Gafsa 2020-08
  [MODIS LST] 600/1104 (54%) — Kebili 2005-12
  [MODIS LST] 700/1104 (63%) — Kebili 2014-04
  [MODIS LST] 800/1104 (72%) — Kebili 2022-08
  [MODIS LST] 900/1104 (82%) — Tozeur 2007-12
  [MODIS LST] 1000/1104 (91%) — Tozeur 2016-04
  [MODIS LST] 1100/1104 (100%) — Tozeur 2024-08
✓ Saved 02_modis_lst_monthly.csv: 1104 rows × 10 cols

Sample:
    governorate  year  month  LST_Day_C_mean  LST_Day_C_max  LST_Day_C_std  \
552      Kebili  2002      1       18.873768      22.554310       2.416870   
553      Kebili  2002      2       25.510021      27.949658       2.065931   
554      Kebili  2002      3       28.818683      31.258908       2.526093   
555      Kebili  2002      4       36.611505      39.858165       2.556347   
556      Kebili  2002      5       39.478845

### Cell 6: 💧 MODIS ET — Evapotranspiration & water stress (HIGH)
**Product**: `MOD16A2.061` (Terra, 500 m, 8-day composites)  
**Bands**: `ET` (actual), `PET` (potential) — scale factor 0.1, units kg/m²/8day  
**Relevance**: The ratio **ETa/PET** is a direct water stress indicator. For irrigated oases,  
a declining ratio signals insufficient irrigation supply — which links directly to aquifer depletion.


In [ ]:
def scale_et(img):
    et  = img.select('ET').multiply(0.1).rename('ETa')
    pet = img.select('PET').multiply(0.1).rename('PET')
    return et.addBands(pet).copyProperties(img, ['system:time_start'])

modis_et = (
    ee.ImageCollection('MODIS/061/MOD16A2')
    .filterDate(START_DATE, END_DATE)
    .map(scale_et)
)

df_et = extract_monthly(
    modis_et, bands=['ETa', 'PET'], gov_geometries=gov_geometries,
    scale=500, description='MODIS ET', save_as='03_modis_et_monthly.csv'
)

print(f"\nSample:\n{df_et[df_et['governorate']=='Gafsa'].head()}")


  [MODIS ET] 100/1104 (9%) — Gabes 2010-04
  [MODIS ET] 200/1104 (18%) — Gabes 2018-08
  [MODIS ET] 300/1104 (27%) — Gafsa 2003-12
  [MODIS ET] 400/1104 (36%) — Gafsa 2012-04
  [MODIS ET] 500/1104 (45%) — Gafsa 2020-08
  [MODIS ET] 600/1104 (54%) — Kebili 2005-12
  [MODIS ET] 700/1104 (63%) — Kebili 2014-04
  [MODIS ET] 800/1104 (72%) — Kebili 2022-08
  [MODIS ET] 900/1104 (82%) — Tozeur 2007-12
  [MODIS ET] 1000/1104 (91%) — Tozeur 2016-04
  [MODIS ET] 1100/1104 (100%) — Tozeur 2024-08
✓ Saved 03_modis_et_monthly.csv: 1104 rows × 10 cols

Sample:
    governorate  year  month  ETa_mean  ETa_max  ETa_std  PET_mean  PET_max  \
276       Gafsa  2002      1       NaN      NaN      NaN       NaN      NaN   
277       Gafsa  2002      2       NaN      NaN      NaN       NaN      NaN   
278       Gafsa  2002      3       NaN      NaN      NaN       NaN      NaN   
279       Gafsa  2002      4       NaN      NaN      NaN       NaN      NaN   
280       Gafsa  2002      5       NaN      NaN    

### Cell 7: 🌡️ ERA5-Land — Climate reanalysis (HIGH)
**Product**: `ECMWF/ERA5_LAND/MONTHLY_AGGR` (9 km, monthly)  
**Bands**: temperature, precipitation, dewpoint, solar radiation, wind  
**Relevance**: The student may already have this extracted. This cell provides a  
verified reference extraction. ERA5-Land at 9 km is the best available reanalysis  
for computing VPD, reference ET, and wind stress (sirocco events).


In [ ]:
era5 = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate(START_DATE, END_DATE)

era5_vars = {
    'temperature_2m':                          'temp_2m_K',
    'total_precipitation_sum':                 'precip_m',
    'dewpoint_temperature_2m':                 'dewpoint_2m_K',
    'surface_solar_radiation_downwards_sum':    'solar_rad_Jm2',
    'u_component_of_wind_10m':                 'wind_u_ms',
    'v_component_of_wind_10m':                 'wind_v_ms',
}

all_era5 = []
for orig_band, short_name in era5_vars.items():
    print(f"\n--- Extracting {short_name} ({orig_band}) ---")
    col = era5.select(orig_band)
    df_band = extract_monthly(
        col, bands=[orig_band], gov_geometries=gov_geometries,
        scale=9000, description=short_name
    )
    # Rename columns to short names
    rename_map = {c: c.replace(orig_band, short_name) for c in df_band.columns if orig_band in c}
    df_band = df_band.rename(columns=rename_map)
    all_era5.append(df_band)

# Merge all bands into one table
keys = ['governorate', 'year', 'month']
df_era5 = all_era5[0]
for df_band in all_era5[1:]:
    new_cols = [c for c in df_band.columns if c not in keys and c != 'image_count' and 'error' not in c]
    df_era5 = df_era5.merge(df_band[keys + new_cols], on=keys, how='outer')

path = os.path.join(DIRS['raw_gee'], '04_era5land_monthly.csv')
df_era5.to_csv(path, index=False)
print(f"\n✓ Saved 04_era5land_monthly.csv: {df_era5.shape}")



--- Extracting temp_2m_K (temperature_2m) ---
  [temp_2m_K] 100/1104 (9%) — Gabes 2010-04
  [temp_2m_K] 200/1104 (18%) — Gabes 2018-08
  [temp_2m_K] 300/1104 (27%) — Gafsa 2003-12
  [temp_2m_K] 400/1104 (36%) — Gafsa 2012-04
  [temp_2m_K] 500/1104 (45%) — Gafsa 2020-08
  [temp_2m_K] 600/1104 (54%) — Kebili 2005-12
  [temp_2m_K] 700/1104 (63%) — Kebili 2014-04
  [temp_2m_K] 800/1104 (72%) — Kebili 2022-08
  [temp_2m_K] 900/1104 (82%) — Tozeur 2007-12
  [temp_2m_K] 1000/1104 (91%) — Tozeur 2016-04
  [temp_2m_K] 1100/1104 (100%) — Tozeur 2024-08

--- Extracting precip_m (total_precipitation_sum) ---
  [precip_m] 100/1104 (9%) — Gabes 2010-04
  [precip_m] 200/1104 (18%) — Gabes 2018-08
  [precip_m] 300/1104 (27%) — Gafsa 2003-12
  [precip_m] 400/1104 (36%) — Gafsa 2012-04
  [precip_m] 500/1104 (45%) — Gafsa 2020-08
  [precip_m] 600/1104 (54%) — Kebili 2005-12
  [precip_m] 700/1104 (63%) — Kebili 2014-04
  [precip_m] 800/1104 (72%) — Kebili 2022-08
  [precip_m] 900/1104 (82%) — Tozeur 2007

### Cell 8: 🌧️ CHIRPS Rainfall (MEDIUM)
**Product**: `UCSB-CHG/CHIRPS/DAILY` (5 km, daily → aggregated to monthly)  
**Variables**: Monthly total (mm), max daily intensity (mm), rain day count  
**Relevance**: Rainfall is minimal in southern Tunisia (<100 mm/yr), but rare events  
can affect pollination success and fruit quality. Also needed for SPI drought indices.


In [ ]:
df_chirps = extract_chirps_monthly(
    gov_geometries, save_as='05_chirps_monthly.csv'
)

print(f"\nSample:\n{df_chirps[df_chirps['governorate']=='Tozeur'].head()}")


  [CHIRPS] 100/1104 (9%)
  [CHIRPS] 200/1104 (18%)
  [CHIRPS] 300/1104 (27%)
  [CHIRPS] 400/1104 (36%)
  [CHIRPS] 500/1104 (45%)
  [CHIRPS] 600/1104 (54%)
  [CHIRPS] 700/1104 (63%)
  [CHIRPS] 800/1104 (72%)
  [CHIRPS] 900/1104 (82%)
  [CHIRPS] 1000/1104 (91%)
  [CHIRPS] 1100/1104 (100%)
✓ Saved 05_chirps_monthly.csv: 1104 rows × 6 cols

Sample:
    governorate  year  month  precip_total_mm  precip_max_daily_mm  rain_days
828      Tozeur  2002      1         6.729708             6.356616   1.111378
829      Tozeur  2002      2         5.054673             4.126188   1.568418
830      Tozeur  2002      3         5.663633             2.389564   3.554825
831      Tozeur  2002      4         3.924710             3.618813   0.965320
832      Tozeur  2002      5        11.410011             6.679476   2.930541


### Cell 9: 🌊 GRACE / GRACE-FO — Groundwater storage (HIGH)
**Product**: `NASA/GRACE/MASS_GRIDS/MASCON_CRI` (JPL mascon, ~0.5°, monthly)  
**Band**: `lwe_thickness` — liquid water equivalent anomaly in cm  
**Relevance**: The only multi-decadal satellite signal of groundwater depletion.  
At ~50 km resolution, one pixel covers the entire study area — so we extract a single  
regional value and assign it to all 4 governorates. Differentiation between governorates  
will come from the CRDA calibration step (Cell 14).

> ⚠ **Data gap**: GRACE ended June 2017, GRACE-FO started June 2018.  
> The ~11-month gap is handled by linear interpolation below.


In [ ]:
# ============================================================
# CELL 9 (CORRECTED): GRACE + GRACE-FO TWS Extraction
# ============================================================
# ISSUE: The original code used only NASA/GRACE/MASS_GRIDS/MASCON_CRI
# which contains GRACE (2002–2017) but NOT GRACE-FO (2018–present).
# GRACE-FO is in a separate GEE collection.
# This version finds and merges both.

print("=" * 60)
print("EXTRACTING: GRACE + GRACE-FO TWS (corrected)")
print("=" * 60)

# --- Step 1: Discover available GRACE/GRACE-FO collections ---
grace_collections = {
    'GRACE (CRI)':      'NASA/GRACE/MASS_GRIDS/MASCON_CRI',
    'GRACE (MASCON)':    'NASA/GRACE/MASS_GRIDS/MASCON',
    'GRACE-FO (CRI v2)':'NASA/GRACE/MASS_GRIDS/MASCON_CRI_RL06_V02',
    'GRACE-FO (v3)':    'NASA/GRACE/MASS_GRIDS/MASCON_CRI_RL06.1_V03',
    'GRACE-FO (alt)':   'NASA/GRACE_FO/MASS_GRIDS/MASCON_CRI_RL06_V02',
    'GRACE-FO (alt v3)':'NASA/GRACE_FO/MASS_GRIDS/MASCON_CRI_RL06.1_V03',
}

found_collections = {}
for name, collection_id in grace_collections.items():
    try:
        col = ee.ImageCollection(collection_id)
        n = col.size().getInfo()
        if n > 0:
            date_range = col.reduceColumns(
                ee.Reducer.minMax(), ['system:time_start']
            ).getInfo()
            from datetime import datetime
            d_min = datetime.fromtimestamp(date_range['min']/1000).strftime('%Y-%m')
            d_max = datetime.fromtimestamp(date_range['max']/1000).strftime('%Y-%m')
            found_collections[name] = {
                'id': collection_id, 'count': n,
                'start': d_min, 'end': d_max
            }
            print(f"  ✓ {name}: {n} images ({d_min} → {d_max})")
        else:
            print(f"  ✗ {name}: empty")
    except Exception as e:
        print(f"  ✗ {name}: not accessible ({str(e)[:60]})")

# --- Step 2: Select best GRACE + GRACE-FO combo ---
# Strategy: find one collection ending before 2018 (GRACE)
#           and one starting after 2017 (GRACE-FO)
grace_id = None
grace_fo_id = None

for name, info in found_collections.items():
    if 'FO' not in name and info['end'] < '2018-01':
        if grace_id is None or info['count'] > found_collections.get(grace_id, {}).get('count', 0):
            grace_id = name
    if 'FO' in name or info['start'] > '2017-06':
        if grace_fo_id is None or info['count'] > found_collections.get(grace_fo_id, {}).get('count', 0):
            grace_fo_id = name

# Also check if any single collection spans the full period
for name, info in found_collections.items():
    if info['start'] <= '2002-06' and info['end'] >= '2023-01':
        print(f"\n  ★ {name} spans full period — using as single source")
        grace_id = name
        grace_fo_id = None
        break

print(f"\n  Selected GRACE:    {grace_id} → {found_collections.get(grace_id, {}).get('id', 'NONE')}")
print(f"  Selected GRACE-FO: {grace_fo_id} → {found_collections.get(grace_fo_id, {}).get('id', 'NONE')}")

if not grace_id:
    raise RuntimeError("No GRACE collection found! Check GEE access.")

# --- Step 3: Load and merge collections ---
band_name = 'lwe_thickness'

# Try to find the right band name
test_col = ee.ImageCollection(found_collections[grace_id]['id'])
test_bands = test_col.first().bandNames().getInfo()
print(f"\n  Available bands: {test_bands}")

if 'lwe_thickness' in test_bands:
    band_name = 'lwe_thickness'
elif 'lwe_thickness_csr' in test_bands:
    band_name = 'lwe_thickness_csr'
elif 'lwe_thickness_jpl' in test_bands:
    band_name = 'lwe_thickness_jpl'
else:
    band_name = test_bands[0]
    print(f"  ⚠ 'lwe_thickness' not found, using first band: {band_name}")

print(f"  Using band: {band_name}")

# Load GRACE
grace_col = (
    ee.ImageCollection(found_collections[grace_id]['id'])
    .filterDate(START_DATE, END_DATE)
    .select(band_name)
)

# Load GRACE-FO if separate
if grace_fo_id:
    grace_fo_col = ee.ImageCollection(found_collections[grace_fo_id]['id'])
    fo_bands = grace_fo_col.first().bandNames().getInfo()
    print(f"  GRACE-FO bands: {fo_bands}")

    fo_band = band_name if band_name in fo_bands else fo_bands[0]
    grace_fo_col = grace_fo_col.filterDate('2018-01-01', END_DATE).select(fo_band)

    # Rename FO band to match GRACE band if different
    if fo_band != band_name:
        grace_fo_col = grace_fo_col.map(
            lambda img: img.rename(band_name).copyProperties(img, ['system:time_start'])
        )

    # Merge
    merged = grace_col.merge(grace_fo_col)
    print(f"\n  Merged: {merged.size().getInfo()} total images")
else:
    merged = grace_col
    print(f"\n  Single collection: {merged.size().getInfo()} images")

# --- Step 4: Extract regional TWS values ---
n_images = merged.size().getInfo()
images_list = merged.sort('system:time_start').toList(n_images)

grace_records = []
for i in range(n_images):
    img = ee.Image(images_list.get(i))
    date = ee.Date(img.get('system:time_start'))
    y = date.get('year').getInfo()
    m = date.get('month').getInfo()

    try:
        val = img.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region_geometry,
            scale=25000, maxPixels=1e9
        ).getInfo()

        grace_records.append({
            'year': y, 'month': m,
            'tws_anomaly_cm': val.get(band_name)
        })
    except Exception as e:
        grace_records.append({'year': y, 'month': m, 'error': str(e)})

    if (i + 1) % 30 == 0:
        print(f"  {i+1}/{n_images} images processed")

df_grace_raw = pd.DataFrame(grace_records)

# --- Step 5: Deduplicate (keep mean of duplicate months) ---
dupes = df_grace_raw.duplicated(subset=['year', 'month'], keep=False)
n_dupes = dupes.sum()
if n_dupes > 0:
    print(f"\n  ⚠ Found {n_dupes} duplicate month entries — averaging them")
    df_grace_raw = (
        df_grace_raw
        .groupby(['year', 'month'])['tws_anomaly_cm']
        .mean()
        .reset_index()
    )

# --- Step 6: Fill gaps with interpolation (only the true gap) ---
df_grace_raw['date'] = pd.to_datetime(
    df_grace_raw[['year', 'month']].assign(day=1)
)

full_dates = pd.date_range(start='2002-01-01', end='2024-12-01', freq='MS')
df_full = pd.DataFrame({'date': full_dates})
df_full = df_full.merge(df_grace_raw[['date', 'tws_anomaly_cm']], on='date', how='left')
df_full['year'] = df_full['date'].dt.year
df_full['month'] = df_full['date'].dt.month

# Mark which values are real vs interpolated
df_full['is_interpolated'] = df_full['tws_anomaly_cm'].isna()
n_missing = df_full['is_interpolated'].sum()
n_total = len(df_full)

print(f"\n  Coverage: {n_total - n_missing}/{n_total} months have real data ({(n_total-n_missing)/n_total*100:.1f}%)")
print(f"  Missing months: {n_missing}")

if n_missing > 0:
    # Show the gap periods
    missing = df_full[df_full['is_interpolated']]
    gap_ranges = []
    prev = None
    start = None
    for _, row in missing.iterrows():
        curr = (row['year'], row['month'])
        if prev is None:
            start = curr
        elif curr[0]*12 + curr[1] > prev[0]*12 + prev[1] + 1:
            gap_ranges.append((start, prev))
            start = curr
        prev = curr
    if start:
        gap_ranges.append((start, prev))

    print("  Gap periods:")
    for (sy, sm), (ey, em) in gap_ranges:
        duration = (ey - sy) * 12 + (em - sm) + 1
        print(f"    {sy}-{sm:02d} → {ey}-{em:02d} ({duration} months)")

    df_full['tws_anomaly_cm'] = df_full['tws_anomaly_cm'].interpolate(method='linear')

# --- Step 7: Expand to all governorates with calibration weights ---
# Load CRDA calibration data
df_crda = pd.read_csv(os.path.join(DIRS['onagri'], 'water_calibration_totals.csv'))

crda_annual = (
    df_crda.groupby(['governorate', 'year'])
    .agg(total_exploitation_Mm3=('exploitation_Mm3', 'sum'))
    .reset_index()
)

mean_exploitation = (
    crda_annual.groupby('governorate')['total_exploitation_Mm3'].mean().reset_index()
)
total_regional = mean_exploitation['total_exploitation_Mm3'].sum()
mean_exploitation['extraction_weight'] = (
    mean_exploitation['total_exploitation_Mm3'] / total_regional
)

weights_dict = dict(zip(
    mean_exploitation['governorate'],
    mean_exploitation['extraction_weight']
))
print(f"\n  Extraction weights: {weights_dict}")

STUDY_AREA_KM2 = 4719 + 22084 + 8990 + 7175
cm_to_Mm3 = STUDY_AREA_KM2 * 0.01

grace_final = []
for gov in GOV_NAMES:
    temp = df_full[['year', 'month', 'tws_anomaly_cm', 'is_interpolated']].copy()
    temp['governorate'] = gov
    w = weights_dict.get(gov, 0.25)
    temp['extraction_weight'] = w
    temp['tws_weighted_cm'] = temp['tws_anomaly_cm'] * w
    temp['tws_estimated_Mm3'] = temp['tws_anomaly_cm'] * cm_to_Mm3 * w
    grace_final.append(temp)

df_grace = pd.concat(grace_final, ignore_index=True)
df_grace = df_grace[['governorate', 'year', 'month', 'tws_anomaly_cm',
                      'is_interpolated', 'extraction_weight',
                      'tws_weighted_cm', 'tws_estimated_Mm3']]

# --- Step 8: Save and summarize ---
path = os.path.join(DIRS['raw_gee'], '06_grace_tws_monthly.csv')
df_grace.to_csv(path, index=False)

# Also save to compiled with calibration
path2 = os.path.join(DIRS['compiled'], 'grace_calibrated_monthly.csv')
df_grace.to_csv(path2, index=False)

print(f"\n✓ Saved 06_grace_tws_monthly.csv: {df_grace.shape}")
print(f"✓ Saved grace_calibrated_monthly.csv: {df_grace.shape}")
print(f"  Expected: {4 * 23 * 12} = 1104 rows | Actual: {len(df_grace)}")
print(f"  TWS range: {df_grace['tws_anomaly_cm'].min():.2f} to {df_grace['tws_anomaly_cm'].max():.2f} cm")

early = df_grace[df_grace['year'] <= 2005]['tws_anomaly_cm'].mean()
late = df_grace[df_grace['year'] >= 2020]['tws_anomaly_cm'].mean()
print(f"  Trend: {early:.2f} cm (2002-05) → {late:.2f} cm (2020-24) = {late-early:+.2f} cm")

real_pct = (1 - df_grace['is_interpolated'].mean()) * 100
print(f"  Real observations: {real_pct:.1f}% | Interpolated: {100-real_pct:.1f}%")

EXTRACTING: GRACE + GRACE-FO TWS (corrected)
  ✓ GRACE (CRI): 163 images (2002-03 → 2017-05)


/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for NASA/GRACE/MASS_GRIDS/MASCON! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/NASA_GRACE_MASS_GRIDS_MASCON

  warnings.warn(warning, category=DeprecationWarning)


  ✓ GRACE (MASCON): 163 images (2002-03 → 2017-05)
  ✗ GRACE-FO (CRI v2): not accessible (ImageCollection.load: ImageCollection asset 'NASA/GRACE/MASS)
  ✗ GRACE-FO (v3): not accessible (ImageCollection.load: ImageCollection asset 'NASA/GRACE/MASS)
  ✗ GRACE-FO (alt): not accessible (ImageCollection.load: ImageCollection asset 'NASA/GRACE_FO/M)
  ✗ GRACE-FO (alt v3): not accessible (ImageCollection.load: ImageCollection asset 'NASA/GRACE_FO/M)

  Selected GRACE:    GRACE (CRI) → NASA/GRACE/MASS_GRIDS/MASCON_CRI
  Selected GRACE-FO: None → NONE

  Available bands: ['lwe_thickness', 'uncertainty']
  Using band: lwe_thickness

  Single collection: 163 images
  30/163 images processed
  60/163 images processed


  90/163 images processed
  120/163 images processed
  150/163 images processed

  ⚠ Found 22 duplicate month entries — averaging them

  Coverage: 152/276 months have real data (55.1%)
  Missing months: 124
  Gap periods:
    2002-01 → 2002-02 (2 months)
    2002-05 → 2002-06 (2 months)
    2003-05 → 2003-05 (1 months)
    2004-01 → 2004-01 (1 months)
    2010-12 → 2011-01 (2 months)
    2011-05 → 2011-06 (2 months)
    2011-11 → 2011-11 (1 months)
    2012-04 → 2012-04 (1 months)
    2012-09 → 2012-10 (2 months)
    2013-02 → 2013-03 (2 months)
    2013-07 → 2013-08 (2 months)
    2014-01 → 2014-02 (2 months)
    2014-06 → 2014-06 (1 months)
    2014-11 → 2014-12 (2 months)
    2015-05 → 2015-05 (1 months)
    2015-09 → 2015-11 (3 months)
    2016-03 → 2016-04 (2 months)
    2016-07 → 2016-07 (1 months)
    2016-09 → 2016-10 (2 months)
    2017-02 → 2017-02 (1 months)
    2017-06 → 2024-12 (91 months)

  Extraction weights: {'Gabes': 0.16917478115471626, 'Gafsa': 0.05631221754593033,

In [ ]:
# ============================================================
# CELL 9 (REPLACEMENT): GRACE + GRACE-FO from NASA PO.DAAC
# ============================================================
#
# WHY: GRACE-FO is NOT available in Google Earth Engine.
# The GEE catalog only contains original GRACE (2002–2017).
# This cell downloads the complete JPL GRACE/GRACE-FO Mascon
# product directly from NASA PO.DAAC, covering 2002–present.
#
# PRODUCT: JPL GRACE/GRACE-FO Mascon RL06.1 v3
#   - CRI-filtered (Coastal Resolution Improvement)
#   - 0.5° × 0.5° grid (~55 km resolution)
#   - Monthly TWS anomaly relative to 2004–2009 baseline
#   - Units: cm of liquid water equivalent
#
# PREREQUISITE: NASA Earthdata account (free)
#   Register at: https://urs.earthdata.nasa.gov/users/new
# ============================================================

# --- Step 0: Install dependencies ---
!pip install -q earthaccess netCDF4 xarray

import earthaccess
import xarray as xr
import numpy as np
import pandas as pd
import os
import json
from datetime import datetime

print("✓ Packages installed")

# ============================================================
# Step 1: Authenticate with NASA Earthdata
# ============================================================
# First run: this will prompt for your Earthdata username/password
# and cache credentials locally. Subsequent runs use the cache.

auth = earthaccess.login(strategy="interactive")

if auth.authenticated:
    print("✓ NASA Earthdata authenticated")
else:
    raise RuntimeError(
        "Authentication failed. Register at https://urs.earthdata.nasa.gov/users/new"
    )

# ============================================================
# Step 2 + 3 (FIXED v4): Direct CMR granule search + download
# ============================================================
# Skip collection discovery — we know the exact product.
# Search granules directly using free_text in CMR.

DOWNLOAD_DIR = os.path.join(
    DIRS.get('raw_gee', '/content/drive/MyDrive/DatePalm_Project/01_raw_gee'),
    'grace_podaac'
)
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- Debug: test what CMR actually returns ---
print("--- Testing CMR search ---")
import requests

# First, find the collection concept_id using free text
cmr_col_url = "https://cmr.earthdata.nasa.gov/search/collections.json"

# Try multiple search strategies
search_attempts = [
    {"provider": "PODAAC", "free_text": "GRACE mascon", "page_size": 20},
    {"provider": "PODAAC", "free_text": "GRACE GRFO mascon", "page_size": 20},
    {"provider": "PODAAC", "short_name": "TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4"},
    {"provider": "PODAAC", "entry_title": "*GRACE*mascon*", "page_size": 20},
    {"free_text": "JPL GRACE GRACE-FO mascon", "page_size": 20},
]

found_cid = None
found_sn = None

for i, params in enumerate(search_attempts):
    resp = requests.get(cmr_col_url, params=params)
    entries = resp.json().get("feed", {}).get("entry", []) if resp.status_code == 200 else []
    print(f"\n  Attempt {i+1} ({list(params.keys())}): {len(entries)} results")

    for entry in entries:
        sn = entry.get("short_name", "")
        title = entry.get("title", "")[:80]
        cid = entry.get("id", "")
        if "MASCON" in sn.upper() or "mascon" in title.lower():
            print(f"    ✓ {sn}: {title}")
            # Prefer CRI + latest version
            if found_cid is None or "CRI" in sn:
                found_cid = cid
                found_sn = sn

    if found_cid:
        break

if found_cid:
    print(f"\n✓ Best match: {found_sn} (concept_id: {found_cid})")
else:
    print("\n  CMR search returned no GRACE mascon results.")
    print("  Trying direct granule download with known URL...\n")

# --- Download approach 1: via earthaccess using concept_id ---
nc_file = None

if found_cid:
    print(f"\n--- Downloading via earthaccess (concept_id: {found_cid}) ---")
    try:
        results = earthaccess.search_data(
            concept_id=found_cid,
            temporal=("2002-01-01", "2024-12-31"),
        )
        if results:
            print(f"  Found {len(results)} granule(s)")
            downloaded = earthaccess.download(results, DOWNLOAD_DIR)
            for f in downloaded:
                fstr = str(f)
                if fstr.endswith('.nc') and os.path.getsize(fstr) > 1024*1024:
                    nc_file = fstr
                    size_mb = os.path.getsize(nc_file) / (1024*1024)
                    print(f"  ✓ Downloaded: {os.path.basename(nc_file)} ({size_mb:.1f} MB)")
                    break
    except Exception as e:
        print(f"  ✗ earthaccess download failed: {e}")

# --- Download approach 2: direct URL with authenticated session ---
if nc_file is None:
    print("\n--- Trying direct authenticated download ---")
    session = earthaccess.get_requests_https_session()

    # Known file paths on PODAAC HTTPS server
    known_urls = [
        "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4/GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04CRI.nc",
        "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4/GRCTellus.JPL.200204_202412.GLO.RL06.3M.MSCNv04CRI.nc",
        "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4/GRCTellus.JPL.200204_202502.GLO.RL06.3M.MSCNv04CRI.nc",
        "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_GRID_RL06.3_V4/GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04.nc",
        "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/TELLUS_GRAC-GRFO_MASCON_GRID_RL06.3_V4/GRCTellus.JPL.200204_202412.GLO.RL06.3M.MSCNv04.nc",
    ]

    for url in known_urls:
        fname = os.path.basename(url)
        local_path = os.path.join(DOWNLOAD_DIR, fname)
        print(f"  Trying: {fname}...")

        try:
            r = session.get(url, stream=True, timeout=120, allow_redirects=True)
            if r.status_code == 200:
                content_type = r.headers.get('content-type', '')
                # Check it's actually a NetCDF and not an HTML error page
                if 'html' not in content_type.lower():
                    total = int(r.headers.get('content-length', 0))
                    downloaded = 0
                    with open(local_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=1024*1024):
                            f.write(chunk)
                            downloaded += len(chunk)
                            if total > 0:
                                print(f"\r    {downloaded/(1024*1024):.0f}/{total/(1024*1024):.0f} MB", end="")
                    size_mb = os.path.getsize(local_path) / (1024*1024)
                    if size_mb > 5:  # Real NetCDF should be >> 5 MB
                        nc_file = local_path
                        print(f"\n  ✓ Downloaded: {fname} ({size_mb:.1f} MB)")
                        break
                    else:
                        os.remove(local_path)
                        print(f"\n    Too small ({size_mb:.1f} MB) — likely error page")
                else:
                    print(f"    Got HTML response — auth redirect or 404")
            else:
                print(f"    HTTP {r.status_code}")
        except Exception as e:
            print(f"    Error: {str(e)[:60]}")

# --- Download approach 3: use OPeNDAP subset (smaller, faster) ---
if nc_file is None:
    print("\n--- Trying OPeNDAP subset download ---")
    # OPeNDAP lets us download just the region we need
    opendap_urls = [
        "https://opendap.earthdata.nasa.gov/providers/PODAAC/collections/GRACE%2FGRACE-FO%20Level-3%20JPL%20RL06.3Mv04%20CRI/granules/GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04CRI",
    ]

    # Subset to our study area: lat 32.5-35, lon 7-11
    subset_params = "?lwe_thickness[0:1:last][140:1:150][374:1:382],lat[140:1:150],lon[374:1:382],time[0:1:last]"

    for base_url in opendap_urls:
        url = base_url + subset_params
        print(f"  Trying OPeNDAP: {base_url.split('/')[-1]}...")
        try:
            ds_remote = xr.open_dataset(base_url, engine='netcdf4')
            print(f"  ✓ OPeNDAP connection successful!")
            print(f"    Variables: {list(ds_remote.data_vars)}")
            print(f"    Time: {len(ds_remote.time)} steps")

            # Save locally
            local_path = os.path.join(DOWNLOAD_DIR, 'grace_gracefo_mascon_subset.nc')
            ds_subset = ds_remote.sel(lat=slice(32.5, 35.0), lon=slice(7.0, 11.0))
            ds_subset.to_netcdf(local_path)
            nc_file = local_path
            size_mb = os.path.getsize(nc_file) / (1024*1024)
            print(f"  ✓ Saved subset: {size_mb:.1f} MB")
            break
        except Exception as e:
            print(f"    ✗ {str(e)[:80]}")

# --- Final check ---
if nc_file is None:
    print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  MANUAL DOWNLOAD REQUIRED                                       ║
║                                                                  ║
║  1. Go to: https://podaac.jpl.nasa.gov/dataset/                ║
║     TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4                 ║
║                                                                  ║
║  2. Click "Data Access" → "Download"                            ║
║     File name looks like:                                        ║
║     GRCTellus.JPL.200204_YYYYMM.GLO.RL06.3M.MSCNv04CRI.nc     ║
║                                                                  ║
║  3. Upload the .nc file to this Google Drive folder:            ║
║     {DOWNLOAD_DIR}                                              ║
║                                                                  ║
║  4. Re-run this cell (it will find the file automatically)      ║
╚══════════════════════════════════════════════════════════════════╝
""")
    # Check if file was manually placed
    existing = [f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.nc')]
    if existing:
        nc_file = os.path.join(DOWNLOAD_DIR, existing[0])
        print(f"  ✓ Found manually uploaded file: {existing[0]}")
    else:
        raise FileNotFoundError("GRACE/GRACE-FO NetCDF not found. Follow manual instructions above.")

print(f"\n{'='*60}")
print(f"✓ NetCDF file ready: {os.path.basename(nc_file)}")
print(f"  Size: {os.path.getsize(nc_file)/(1024*1024):.1f} MB")
# ============================================================
# Step 4: Open and explore the dataset
# ============================================================

print("\n--- Dataset structure ---")
ds = xr.open_dataset(nc_file)
print(f"  Dimensions: {dict(ds.dims)}")
print(f"  Variables: {list(ds.data_vars)}")
print(f"  Time range: {str(ds.time.values[0])[:10]} → {str(ds.time.values[-1])[:10]}")
print(f"  Time steps: {len(ds.time)}")

# Identify the TWS variable
tws_var = None
for candidate in ['lwe_thickness', 'lwe_thickness_cri', 'lwe_thickness_jpl',
                   'mascon', 'twsa', 'Liquid_Water_Equivalent_Thickness']:
    if candidate in ds.data_vars:
        tws_var = candidate
        break

if tws_var is None:
    # Use first non-coordinate variable
    data_vars = [v for v in ds.data_vars if 'time' in ds[v].dims]
    if data_vars:
        tws_var = data_vars[0]
    else:
        raise ValueError(f"Cannot identify TWS variable. Available: {list(ds.data_vars)}")

print(f"  TWS variable: '{tws_var}'")
print(f"  Units: {ds[tws_var].attrs.get('units', 'unknown')}")
print(f"  Shape: {ds[tws_var].shape}")

# Check coordinate names
lat_name = 'lat' if 'lat' in ds.coords else 'latitude'
lon_name = 'lon' if 'lon' in ds.coords else 'longitude'
print(f"  Lat coord: '{lat_name}' ({float(ds[lat_name].min()):.1f}° to {float(ds[lat_name].max()):.1f}°)")
print(f"  Lon coord: '{lon_name}' ({float(ds[lon_name].min()):.1f}° to {float(ds[lon_name].max()):.1f}°)")

# ============================================================
# Step 5: Extract TWS for study area
# ============================================================

# Southern Tunisia bounding box
# Tozeur: ~33.9°N, 8.1°E | Kebili: ~33.7°N, 8.9°E
# Gafsa: ~34.4°N, 8.8°E  | Gabes: ~33.9°N, 10.1°E
LAT_MIN, LAT_MAX = 32.5, 35.0
LON_MIN, LON_MAX = 7.0, 11.0

print(f"\n--- Extracting for Southern Tunisia ---")
print(f"  Bounding box: [{LAT_MIN}°N–{LAT_MAX}°N] × [{LON_MIN}°E–{LON_MAX}°E]")

# Slice to study area
ds_region = ds.sel(**{
    lat_name: slice(
        min(LAT_MIN, LAT_MAX) if ds[lat_name].values[0] < ds[lat_name].values[-1] else max(LAT_MIN, LAT_MAX),
        max(LAT_MIN, LAT_MAX) if ds[lat_name].values[0] < ds[lat_name].values[-1] else min(LAT_MIN, LAT_MAX)
    ),
    lon_name: slice(LON_MIN, LON_MAX)
})

n_pixels = ds_region[lat_name].size * ds_region[lon_name].size
print(f"  Grid cells in region: {ds_region[lat_name].size} lat × {ds_region[lon_name].size} lon = {n_pixels} pixels")

# Compute area-weighted regional mean TWS per month
# Weight by cosine of latitude (accounts for grid cell area variation)
weights = np.cos(np.deg2rad(ds_region[lat_name]))
tws_regional = ds_region[tws_var].weighted(weights).mean(dim=[lat_name, lon_name])

# Convert to DataFrame
df_tws = tws_regional.to_dataframe(name='tws_anomaly_cm').reset_index()
df_tws['year'] = pd.to_datetime(df_tws['time']).dt.year
df_tws['month'] = pd.to_datetime(df_tws['time']).dt.month

# Filter to 2002–2024
df_tws = df_tws[(df_tws['year'] >= 2002) & (df_tws['year'] <= 2024)].copy()
df_tws = df_tws[['year', 'month', 'tws_anomaly_cm']].reset_index(drop=True)

print(f"\n  Extracted {len(df_tws)} monthly TWS values")
print(f"  Year range: {df_tws['year'].min()}–{df_tws['year'].max()}")
print(f"  TWS range: {df_tws['tws_anomaly_cm'].min():.2f} to {df_tws['tws_anomaly_cm'].max():.2f} cm")

# ============================================================
# Step 6: Identify real data vs gaps
# ============================================================

print(f"\n--- Coverage analysis ---")

# Deduplicate if needed (keep mean)
dupes = df_tws.duplicated(subset=['year', 'month'], keep=False)
if dupes.any():
    n_dupes = dupes.sum()
    print(f"  ⚠ {n_dupes} duplicate months found — averaging")
    df_tws = df_tws.groupby(['year', 'month'])['tws_anomaly_cm'].mean().reset_index()

# Build complete monthly index
full_dates = pd.date_range(start='2002-01-01', end='2024-12-01', freq='MS')
df_full = pd.DataFrame({
    'year': full_dates.year,
    'month': full_dates.month,
})

df_full = df_full.merge(df_tws, on=['year', 'month'], how='left')
df_full['is_interpolated'] = df_full['tws_anomaly_cm'].isna()

n_real = df_full['tws_anomaly_cm'].notna().sum()
n_total = len(df_full)
n_missing = n_total - n_real

print(f"  Real observations: {n_real}/{n_total} ({n_real/n_total*100:.1f}%)")
print(f"  Missing months: {n_missing}")

# Show gap periods
if n_missing > 0:
    missing_rows = df_full[df_full['is_interpolated']]
    gap_ranges = []
    prev_idx = None
    start_row = None

    for _, row in missing_rows.iterrows():
        curr_idx = row['year'] * 12 + row['month']
        if prev_idx is None:
            start_row = row
        elif curr_idx > prev_idx + 1:
            gap_ranges.append((
                f"{int(start_row['year'])}-{int(start_row['month']):02d}",
                f"{int(prev_row['year'])}-{int(prev_row['month']):02d}",
                curr_idx - (int(start_row['year'])*12 + int(start_row['month']))
            ))
            start_row = row
        prev_row = row
        prev_idx = curr_idx

    if start_row is not None:
        gap_ranges.append((
            f"{int(start_row['year'])}-{int(start_row['month']):02d}",
            f"{int(prev_row['year'])}-{int(prev_row['month']):02d}",
            int(prev_row['year'])*12 + int(prev_row['month']) - int(start_row['year'])*12 - int(start_row['month']) + 1
        ))

    print("\n  Gap periods:")
    for start, end, duration in gap_ranges:
        label = " ← GRACE→GRACE-FO transition" if "2017" in start or "2018" in start else ""
        print(f"    {start} → {end} ({duration} months){label}")

    # Interpolate gaps
    df_full['tws_anomaly_cm'] = df_full['tws_anomaly_cm'].interpolate(method='linear')
    print(f"\n  ✓ Gaps filled with linear interpolation")

# ============================================================
# Step 7: Validate — check GRACE vs GRACE-FO continuity
# ============================================================

print(f"\n--- GRACE vs GRACE-FO comparison ---")

grace_era = df_full[(df_full['year'] >= 2002) & (df_full['year'] <= 2016) & ~df_full['is_interpolated']]
gracefo_era = df_full[(df_full['year'] >= 2019) & ~df_full['is_interpolated']]

if len(grace_era) > 0:
    print(f"  GRACE (2002–2016):    {len(grace_era)} months, "
          f"mean TWS = {grace_era['tws_anomaly_cm'].mean():.2f} cm")
if len(gracefo_era) > 0:
    print(f"  GRACE-FO (2019–2024): {len(gracefo_era)} months, "
          f"mean TWS = {gracefo_era['tws_anomaly_cm'].mean():.2f} cm")
    print(f"  ✓ GRACE-FO data is REAL (not interpolated)")
else:
    print(f"  ⛔ NO GRACE-FO data found — check the downloaded file!")
    print(f"     This means the NetCDF only contains original GRACE (2002–2017)")

# Annual trend
annual_tws = df_full.groupby('year')['tws_anomaly_cm'].mean()
early = annual_tws[annual_tws.index <= 2005].mean()
late = annual_tws[annual_tws.index >= 2020].mean()
print(f"\n  Depletion trend: {early:.2f} cm (2002–05) → {late:.2f} cm (2020–24) = {late-early:+.2f} cm")

# ============================================================
# Step 8: Apply CRDA calibration & expand to governorates
# ============================================================

print(f"\n--- Applying CRDA calibration ---")

# Load CRDA calibration data
crda_path = os.path.join(
    DIRS.get('onagri', '/content/drive/MyDrive/DatePalm_Project/02_onagri'),
    'water_calibration_totals.csv'
)
df_crda = pd.read_csv(crda_path)

# Compute extraction weights
crda_annual = (
    df_crda.groupby(['governorate', 'year'])
    .agg(total_exploitation_Mm3=('exploitation_Mm3', 'sum'))
    .reset_index()
)
mean_exploitation = (
    crda_annual.groupby('governorate')['total_exploitation_Mm3'].mean().reset_index()
)
total_regional = mean_exploitation['total_exploitation_Mm3'].sum()
mean_exploitation['extraction_weight'] = (
    mean_exploitation['total_exploitation_Mm3'] / total_regional
)
weights_dict = dict(zip(
    mean_exploitation['governorate'],
    mean_exploitation['extraction_weight']
))

# Study area for cm → Mm³ conversion
STUDY_AREA_KM2 = 4719 + 22084 + 8990 + 7175  # ~42,968 km²
cm_to_Mm3 = STUDY_AREA_KM2 * 0.01

print(f"  Extraction weights:")
for gov, w in weights_dict.items():
    bar = "█" * int(w * 50)
    print(f"    {gov:<10}: {w:.3f} {bar}")
print(f"  Scale: 1 cm TWS ≈ {cm_to_Mm3:.1f} Mm³ over study area")

# Expand to governorates
grace_final = []
GOV_NAMES_LOCAL = sorted(weights_dict.keys())

for gov in GOV_NAMES_LOCAL:
    temp = df_full[['year', 'month', 'tws_anomaly_cm', 'is_interpolated']].copy()
    temp['governorate'] = gov
    w = weights_dict[gov]
    temp['extraction_weight'] = w
    temp['tws_weighted_cm'] = temp['tws_anomaly_cm'] * w
    temp['tws_estimated_Mm3'] = temp['tws_anomaly_cm'] * cm_to_Mm3 * w
    grace_final.append(temp)

df_grace = pd.concat(grace_final, ignore_index=True)
df_grace = df_grace[['governorate', 'year', 'month', 'tws_anomaly_cm',
                      'is_interpolated', 'extraction_weight',
                      'tws_weighted_cm', 'tws_estimated_Mm3']]

# ============================================================
# Step 9: Save outputs
# ============================================================

# Save to raw_gee
raw_path = os.path.join(
    DIRS.get('raw_gee', '/content/drive/MyDrive/DatePalm_Project/01_raw_gee'),
    '06_grace_tws_monthly.csv'
)
df_grace.to_csv(raw_path, index=False)

# Save to compiled (with calibration)
compiled_path = os.path.join(
    DIRS.get('compiled', '/content/drive/MyDrive/DatePalm_Project/03_compiled'),
    'grace_calibrated_monthly.csv'
)
df_grace.to_csv(compiled_path, index=False)

# Save calibration metadata
calib_doc = {
    'source': 'NASA PO.DAAC JPL GRACE/GRACE-FO Mascon RL06.1 v3',
    'download_date': datetime.now().isoformat(),
    'nc_file': os.path.basename(nc_file),
    'study_area_bbox': {'lat': [LAT_MIN, LAT_MAX], 'lon': [LON_MIN, LON_MAX]},
    'study_area_km2': STUDY_AREA_KM2,
    'cm_to_Mm3_factor': cm_to_Mm3,
    'extraction_weights': weights_dict,
    'total_months': len(df_full),
    'real_observations': int(n_real),
    'interpolated_months': int(n_missing),
    'interpolation_pct': round(n_missing / n_total * 100, 1),
    'methodology': (
        'Area-weighted mean TWS anomaly over study region bbox. '
        'Cosine-latitude weighting applied. '
        'Governorate differentiation via CRDA extraction weight scaling. '
        'Missing months filled with linear interpolation.'
    ),
}
doc_path = os.path.join(
    DIRS.get('docs', '/content/drive/MyDrive/DatePalm_Project/04_documentation'),
    'grace_calibration_metadata.json'
)
with open(doc_path, 'w') as f:
    json.dump(calib_doc, f, indent=2, default=str)

# ============================================================
# Final summary
# ============================================================

print(f"\n{'='*60}")
print(f"GRACE/GRACE-FO EXTRACTION COMPLETE")
print(f"{'='*60}")
print(f"  Source:        NASA PO.DAAC (JPL Mascon RL06.1)")
print(f"  Rows:          {len(df_grace)} (expected {4*23*12} = 1104)")
print(f"  Columns:       {list(df_grace.columns)}")
print(f"  Real data:     {n_real}/{n_total} months ({n_real/n_total*100:.1f}%)")
print(f"  Interpolated:  {n_missing}/{n_total} months ({n_missing/n_total*100:.1f}%)")
print(f"  TWS range:     {df_grace['tws_anomaly_cm'].min():.2f} to {df_grace['tws_anomaly_cm'].max():.2f} cm")
print(f"  Trend:         {early:.2f} → {late:.2f} cm ({late-early:+.2f} cm depletion)")
print(f"\n  Saved to:")
print(f"    {raw_path}")
print(f"    {compiled_path}")
print(f"    {doc_path}")
print(f"\n  ⚠ After running this cell, re-run Cells 15–16 to rebuild the master datasets")
print(f"     then Cells 17–19 to re-verify data quality")

✓ Packages installed
✓ NASA Earthdata authenticated
--- Testing CMR search ---

  Attempt 1 (['provider', 'free_text', 'page_size']): 0 results

  Attempt 2 (['provider', 'free_text', 'page_size']): 0 results

  Attempt 3 (['provider', 'short_name']): 0 results

  Attempt 4 (['provider', 'entry_title', 'page_size']): 0 results

  Attempt 5 (['free_text', 'page_size']): 0 results

  CMR search returned no GRACE mascon results.
  Trying direct granule download with known URL...


--- Trying direct authenticated download ---
  Trying: GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04CRI.nc...
    41/41 MB
  ✓ Downloaded: GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04CRI.nc (40.9 MB)

✓ NetCDF file ready: GRCTellus.JPL.200204_202501.GLO.RL06.3M.MSCNv04CRI.nc
  Size: 40.9 MB

--- Dataset structure ---
  Dimensions: {'time': 241, 'lat': 360, 'lon': 720, 'bounds': 2}
  Variables: ['lwe_thickness', 'uncertainty', 'lat_bounds', 'lon_bounds', 'time_bounds', 'land_mask', 'scale_factor', 'mascon_ID

### Cell 10: 🛰️ Sentinel-2 NDVI + NDWI (2015–2024 only) (MEDIUM)
**Product**: `COPERNICUS/S2_SR_HARMONIZED` (10 m, 5-day revisit)  
**Computed indices**:
- **NDVI** = (B8 − B4) / (B8 + B4) — vegetation vigor  
- **NDWI** = (B3 − B8) / (B3 + B8) — vegetation water content  

**Relevance**: Higher resolution than MODIS for the recent 10-year period.  
NDWI is particularly valuable — it directly indicates plant water stress,  
which in oasis systems links to irrigation adequacy and aquifer supply.


In [ ]:
S2_START = '2015-07-01'
S2_YEARS = list(range(2015, 2025))

def mask_s2_clouds(img):
    qa = img.select('QA60')
    mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return img.updateMask(mask)

def add_s2_indices(img):
    ndvi = img.normalizedDifference(['B8', 'B4']).rename('S2_NDVI')
    ndwi = img.normalizedDifference(['B3', 'B8']).rename('S2_NDWI')
    return ndvi.addBands(ndwi).copyProperties(img, ['system:time_start'])

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(S2_START, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
    .map(add_s2_indices)
)

# Custom extraction — Sentinel-2 needs filterBounds per governorate
results = []
total = len(gov_geometries) * len(S2_YEARS) * 12
counter = 0

for gov_name, geom in gov_geometries.items():
    for year in S2_YEARS:
        for month in MONTHS:
            counter += 1
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')

            monthly = s2_collection.filterDate(start, end).filterBounds(geom)
            row = {'governorate': gov_name, 'year': year, 'month': month}

            try:
                n = monthly.size().getInfo()
                row['S2_scene_count'] = n
                if n == 0:
                    row.update({'S2_NDVI_mean': None, 'S2_NDWI_mean': None})
                else:
                    composite = monthly.median()
                    vals = composite.reduceRegion(
                        reducer=ee.Reducer.mean().combine(
                            ee.Reducer.stdDev(), sharedInputs=True
                        ),
                        geometry=geom, scale=100, maxPixels=1e9
                    ).getInfo()
                    row['S2_NDVI_mean'] = vals.get('S2_NDVI_mean')
                    row['S2_NDVI_std']  = vals.get('S2_NDVI_stdDev')
                    row['S2_NDWI_mean'] = vals.get('S2_NDWI_mean')
                    row['S2_NDWI_std']  = vals.get('S2_NDWI_stdDev')
            except Exception as e:
                row['error'] = str(e)

            results.append(row)
            if counter % 50 == 0:
                print(f"  [Sentinel-2] {counter}/{total} ({counter/total*100:.0f}%)")

df_s2 = pd.DataFrame(results)
path = os.path.join(DIRS['raw_gee'], '07_sentinel2_ndvi_ndwi_monthly.csv')
df_s2.to_csv(path, index=False)
print(f"✓ Saved 07_sentinel2_ndvi_ndwi_monthly.csv: {df_s2.shape}")


  [Sentinel-2] 50/480 (10%)
  [Sentinel-2] 100/480 (21%)
  [Sentinel-2] 150/480 (31%)
  [Sentinel-2] 200/480 (42%)
  [Sentinel-2] 250/480 (52%)


  [Sentinel-2] 300/480 (62%)
  [Sentinel-2] 350/480 (73%)
  [Sentinel-2] 400/480 (83%)
  [Sentinel-2] 450/480 (94%)
✓ Saved 07_sentinel2_ndvi_ndwi_monthly.csv: (480, 8)


---
## Phase C — Static Spatial Data
### Cell 11: 🏔️ SoilGrids + SRTM DEM — One-time extraction
These are time-invariant features that differentiate the four governorates.  
They serve as **static covariates** in the model (especially useful for pooled models).


In [ ]:
srtm = ee.Image('USGS/SRTMGL1_003').select('elevation')

# SoilGrids — try the ISRIC collection
try:
    soil_clay = ee.Image("projects/soilgrids-isric/clay_mean").select('clay_0-5cm_mean')
    soil_sand = ee.Image("projects/soilgrids-isric/sand_mean").select('sand_0-5cm_mean')
    soil_soc  = ee.Image("projects/soilgrids-isric/soc_mean").select('soc_0-5cm_mean')
    has_soil = True
except:
    print("⚠ SoilGrids not accessible in GEE. Will skip soil variables.")
    has_soil = False

static_records = []
for gov_name, geom in gov_geometries.items():
    print(f"  Processing {gov_name}...")
    row = {'governorate': gov_name}

    try:
        elev = srtm.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True)
                    .combine(ee.Reducer.minMax(), sharedInputs=True),
            geometry=geom, scale=90, maxPixels=1e9
        ).getInfo()
        row['elevation_mean_m'] = elev.get('elevation_mean')
        row['elevation_std_m']  = elev.get('elevation_stdDev')
        row['elevation_min_m']  = elev.get('elevation_min')
        row['elevation_max_m']  = elev.get('elevation_max')
    except Exception as e:
        row['elev_error'] = str(e)

    if has_soil:
        for img, name in [(soil_clay, 'clay_pct'), (soil_sand, 'sand_pct'),
                           (soil_soc, 'soc_g_kg')]:
            try:
                val = img.reduceRegion(
                    reducer=ee.Reducer.mean(), geometry=geom,
                    scale=250, maxPixels=1e9
                ).getInfo()
                row[name] = list(val.values())[0]
            except:
                row[name] = None

    static_records.append(row)

df_static = pd.DataFrame(static_records)
path = os.path.join(DIRS['raw_gee'], '08_static_spatial.csv')
df_static.to_csv(path, index=False)
print(f"\n✓ Saved 08_static_spatial.csv:")
print(df_static.to_string(index=False))


  Processing Gabes...
  Processing Gafsa...
  Processing Kebili...
  Processing Tozeur...

✓ Saved 08_static_spatial.csv:
governorate  elevation_mean_m  elevation_std_m  elevation_min_m  elevation_max_m   clay_pct   sand_pct   soc_g_kg
      Gabes        146.431036       123.102248              -34              694 197.907118 523.569282  92.852984
      Gafsa        361.553699       188.025065                8             1153 198.304466 497.310387  91.442167
     Kebili        107.907260        82.337298              -30              618 238.213955 426.152934  82.762512
     Tozeur         46.488534       104.290195              -57              860 242.266477 451.853569 125.425768


---
## Phase D — ONAGRI Ground Truth Integration
### Cell 12: Upload ONAGRI clean dataset

Upload the **`onagri_clean_dataset.xlsx`** file prepared in Phase 1.  
This contains:
- **production_target**: 92 rows — yearly production in tonnes per governorate (2002–2024)
- **water_totals**: 16 rows — aquifer exploitation summaries (calibration anchors)
- **water_full_detail**: 131 rows — per-aquifer breakdowns with well counts


In [ ]:
# --- Read ONAGRI dataset directly from Drive ---
onagri_file = os.path.join(DIRS['onagri'], 'onagri_clean_dataset.xlsx')

if not os.path.exists(onagri_file):
    raise FileNotFoundError(
        f"ONAGRI file not found at: {onagri_file}\n"
        f"Please upload 'onagri_clean_dataset.xlsx' to your Google Drive at:\n"
        f"  Google Drive > DatePalm_Project > 02_onagri > onagri_clean_dataset.xlsx\n"
        f"Then re-run this cell."
    )

print(f"✓ Found: {onagri_file}")

# Read all sheets
df_production = pd.read_excel(onagri_file, sheet_name='production_target')
df_water_totals = pd.read_excel(onagri_file, sheet_name='water_totals')
df_water_detail = pd.read_excel(onagri_file, sheet_name='water_full_detail')

# Save as individual CSVs for downstream cells
for df, name in [(df_production, 'production_target.csv'),
                  (df_water_totals, 'water_calibration_totals.csv'),
                  (df_water_detail, 'water_calibration_full.csv')]:
    df.to_csv(os.path.join(DIRS['onagri'], name), index=False)

print(f"\n✓ Production data: {df_production.shape}")
print(df_production.groupby('governorate')['year'].agg(['count','min','max']))

print(f"\n✓ Water calibration totals: {df_water_totals.shape}")
print(df_water_totals[['governorate','year','aquifer_type',
                        'resources_Mm3','exploitation_Mm3','exploitation_rate_pct']].to_string(index=False))


✓ Found: /content/drive/MyDrive/DatePalm_Project/02_onagri/onagri_clean_dataset.xlsx

✓ Production data: (92, 3)
             count   min   max
governorate                   
Gabes           23  2002  2024
Gafsa           23  2002  2024
Kebili          23  2002  2024
Tozeur          23  2002  2024

✓ Water calibration totals: (16, 8)
governorate  year  aquifer_type  resources_Mm3  exploitation_Mm3  exploitation_rate_pct
      Gabes  2024    deep_total         155.80            135.30              86.842105
      Gabes  2024 shallow_total          24.38             28.10             115.258409
      Gafsa  2023 shallow_total          33.30             54.39             163.333333
      Gafsa  2024 shallow_total          33.30             54.39             163.333333
     Kebili  2022    deep_total         236.70            539.58             227.959442
     Kebili  2022 shallow_total           5.49              0.17               3.096539
     Kebili  2023    deep_total         236.70  

### Cell 13: Verify all collected data files


In [ ]:
print("=" * 70)
print("DATA INVENTORY CHECK")
print("=" * 70)

expected_files = {
    'raw_gee': [
        '01_modis_ndvi_monthly.csv',
        '02_modis_lst_monthly.csv',
        '03_modis_et_monthly.csv',
        '04_era5land_monthly.csv',
        '05_chirps_monthly.csv',
        '06_grace_tws_monthly.csv',
        '07_sentinel2_ndvi_ndwi_monthly.csv',
        '08_static_spatial.csv',
    ],
    'onagri': [
        'production_target.csv',
        'water_calibration_totals.csv',
        'water_calibration_full.csv',
    ]
}

all_ok = True
for folder, filenames in expected_files.items():
    print(f"\n📁 {DIRS[folder]}/")
    for f in filenames:
        path = os.path.join(DIRS[folder], f)
        if os.path.exists(path):
            df = pd.read_csv(path)
            print(f"  ✓ {f:<45} {df.shape[0]:>5} rows × {df.shape[1]:>3} cols")
        else:
            print(f"  ✗ {f:<45} MISSING")
            all_ok = False

if all_ok:
    print("\n✅ All files present — ready for GRACE calibration and compilation")
else:
    print("\n⚠ Some files are missing. Re-run the corresponding extraction cells.")


DATA INVENTORY CHECK

📁 /content/drive/MyDrive/DatePalm_Project/01_raw_gee/
  ✓ 01_modis_ndvi_monthly.csv                      1104 rows ×   7 cols
  ✓ 02_modis_lst_monthly.csv                       1104 rows ×  10 cols
  ✓ 03_modis_et_monthly.csv                        1104 rows ×  10 cols
  ✓ 04_era5land_monthly.csv                        1104 rows ×  22 cols
  ✓ 05_chirps_monthly.csv                          1104 rows ×   6 cols
  ✓ 06_grace_tws_monthly.csv                       1148 rows ×   5 cols
  ✓ 07_sentinel2_ndvi_ndwi_monthly.csv              480 rows ×   8 cols
  ✓ 08_static_spatial.csv                             4 rows ×   8 cols

📁 /content/drive/MyDrive/DatePalm_Project/02_onagri/
  ✓ production_target.csv                            92 rows ×   3 cols
  ✓ water_calibration_totals.csv                     16 rows ×   8 cols
  ✓ water_calibration_full.csv                      131 rows ×   9 cols

✅ All files present — ready for GRACE calibration and compilation


---
## Phase E — GRACE/GRACE-FO TWS Calibration

### Background & Methodology

GRACE provides **Terrestrial Water Storage (TWS) anomaly** — the total change in water  
stored in all reservoirs (soil moisture, surface water, groundwater) relative to a baseline  
period (2004–2009 mean). In southern Tunisia, there is virtually no surface water and  
soil moisture is minimal, so **TWS ≈ groundwater storage change**.

However, GRACE has two critical limitations:
1. **Coarse resolution** (~50 km) — cannot distinguish between governorates
2. **Anomaly, not absolute volume** — reports change in cm of water equivalent, not Mm³

**Calibration approach**: We use the CRDA ground data (absolute exploitation volumes in Mm³)  
from 2020–2024 to establish a **linear relationship** between GRACE TWS anomaly and  
actual extraction intensity per governorate. This allows us to:
- Convert TWS anomaly (cm) → estimated extraction stress per governorate
- Extend the CRDA-like signal backwards to 2002 using the GRACE time series

### Mathematical formulation

For each governorate *g* and calibration year *t*:

```
GRACE_TWS(t)  — regional satellite signal (cm)
EXPL_RATIO(g,t) = exploitation(g,t) / resources(g,t) — from CRDA
```

We compute a **governorate-specific weight** based on each governorate's share of  
total regional extraction:

```
weight(g,t) = exploitation(g,t) / Σ exploitation(all_govs, t)
```

Then the calibrated TWS for governorate *g* at any month is:

```
TWS_calibrated(g, month) = GRACE_TWS(month) × weight(g) × scale_factor
```

Where `scale_factor` converts cm → Mm³ using the regional area and known extraction totals.

### Cell 14: GRACE Calibration


In [ ]:
# ============================================================
# Step 1: Load GRACE and CRDA data
# ============================================================

df_grace = pd.read_csv(os.path.join(DIRS['raw_gee'], '06_grace_tws_monthly.csv'))
df_crda = pd.read_csv(os.path.join(DIRS['onagri'], 'water_calibration_totals.csv'))

print("=== CRDA Calibration Data ===")
print(df_crda[['governorate','year','aquifer_type','resources_Mm3',
               'exploitation_Mm3','exploitation_rate_pct']].to_string(index=False))

# ============================================================
# Step 2: Compute total groundwater extraction per governorate-year
# Sum shallow + deep for each governorate where we have data
# ============================================================

crda_annual = (
    df_crda
    .groupby(['governorate', 'year'])
    .agg(
        total_resources_Mm3=('resources_Mm3', 'sum'),
        total_exploitation_Mm3=('exploitation_Mm3', 'sum'),
    )
    .reset_index()
)
crda_annual['exploitation_ratio'] = (
    crda_annual['total_exploitation_Mm3'] / crda_annual['total_resources_Mm3']
)

print("\n=== Annual Totals (shallow + deep combined) ===")
print(crda_annual.to_string(index=False))

# ============================================================
# Step 3: Compute governorate extraction weights
# For years with overlapping data across governorates, compute
# each gov's share of total regional extraction
# ============================================================

# Use the most complete overlapping year (check what's available)
overlap_years = crda_annual.groupby('year')['governorate'].count()
print(f"\n=== Governorates with data per year ===\n{overlap_years}")

# Use all available data — compute mean exploitation per gov
mean_exploitation = (
    crda_annual
    .groupby('governorate')['total_exploitation_Mm3']
    .mean()
    .reset_index()
)
total_regional = mean_exploitation['total_exploitation_Mm3'].sum()
mean_exploitation['extraction_weight'] = (
    mean_exploitation['total_exploitation_Mm3'] / total_regional
)

print(f"\n=== Governorate Extraction Weights ===")
print(mean_exploitation.to_string(index=False))
print(f"\nTotal regional extraction: {total_regional:.1f} Mm³/yr")

# ============================================================
# Step 4: Compute TWS → Mm³ scale factor
# GRACE TWS is in cm of water equivalent over the study area
# Convert: 1 cm over area A (km²) = A × 0.01 × 1e6 m² × 0.01 m = A × 1e-4 Mm³
# ============================================================

# Approximate total study area (4 governorates combined)
# Tozeur: ~4,719 km², Kebili: ~22,084 km², Gafsa: ~8,990 km², Gabes: ~7,175 km²
STUDY_AREA_KM2 = 4719 + 22084 + 8990 + 7175  # ~42,968 km²

# 1 cm of water over study area = STUDY_AREA_KM2 * 1e6 m² * 0.01 m / 1e6 = km² * 0.01 Mm³
cm_to_Mm3 = STUDY_AREA_KM2 * 0.01  # ~429.7 Mm³ per cm

print(f"\nStudy area: {STUDY_AREA_KM2:,} km²")
print(f"Scale factor: 1 cm TWS ≈ {cm_to_Mm3:.1f} Mm³")

# ============================================================
# Step 5: Apply calibration — create governorate-specific TWS
# ============================================================

# Get GRACE annual mean TWS for calibration years
grace_annual = (
    df_grace[df_grace['governorate'] == GOV_NAMES[0]]  # Same value for all govs
    .groupby('year')['tws_anomaly_cm']
    .mean()
    .reset_index()
    .rename(columns={'tws_anomaly_cm': 'tws_annual_mean_cm'})
)

# Merge weights into the monthly GRACE data
weights_dict = dict(zip(
    mean_exploitation['governorate'],
    mean_exploitation['extraction_weight']
))

df_grace['extraction_weight'] = df_grace['governorate'].map(weights_dict)

# Calibrated TWS per governorate:
# - tws_weighted_cm: governorate-weighted anomaly
# - tws_estimated_Mm3: estimated volume change attributed to this governorate
df_grace['tws_weighted_cm'] = df_grace['tws_anomaly_cm'] * df_grace['extraction_weight']
df_grace['tws_estimated_Mm3'] = df_grace['tws_anomaly_cm'] * cm_to_Mm3 * df_grace['extraction_weight']

# ============================================================
# Step 6: Validate against CRDA data where they overlap
# ============================================================

print("\n=== Calibration Validation ===")
print("Comparing GRACE-derived estimates with CRDA actuals for overlapping years:\n")

for _, row in crda_annual.iterrows():
    gov = row['governorate']
    yr = row['year']
    crda_val = row['total_exploitation_Mm3']

    grace_yr = df_grace[
        (df_grace['governorate'] == gov) & (df_grace['year'] == yr)
    ]['tws_estimated_Mm3'].mean()

    print(f"  {gov} {yr}: CRDA = {crda_val:.1f} Mm³ | "
          f"GRACE-derived ≈ {abs(grace_yr):.1f} Mm³ "
          f"(ratio: {abs(grace_yr)/crda_val:.2f})")

print("\n⚠ Note: GRACE measures CHANGE in storage, not total extraction.")
print("  The absolute values won't match. What matters is the TREND correlation")
print("  and the relative differences between governorates.")

# ============================================================
# Step 7: Save calibrated GRACE data
# ============================================================

# Save full calibrated dataset
path = os.path.join(DIRS['compiled'], 'grace_calibrated_monthly.csv')
df_grace.to_csv(path, index=False)
print(f"\n✓ Saved grace_calibrated_monthly.csv: {df_grace.shape}")

# Save calibration documentation
calib_doc = {
    'study_area_km2': STUDY_AREA_KM2,
    'cm_to_Mm3_factor': cm_to_Mm3,
    'extraction_weights': weights_dict,
    'calibration_years': crda_annual[['governorate','year']].to_dict('records'),
    'methodology': 'Linear weight-based downscaling of GRACE TWS anomaly',
    'limitation': 'GRACE measures total water storage change, not extraction volume. '
                  'Weights assume stable relative extraction shares across time.',
    'generated': datetime.now().isoformat()
}
doc_path = os.path.join(DIRS['docs'], 'grace_calibration_metadata.json')
with open(doc_path, 'w') as f:
    json.dump(calib_doc, f, indent=2, default=str)
print(f"✓ Saved calibration documentation: {doc_path}")


=== CRDA Calibration Data ===
governorate  year  aquifer_type  resources_Mm3  exploitation_Mm3  exploitation_rate_pct
      Gabes  2024    deep_total         155.80            135.30              86.842105
      Gabes  2024 shallow_total          24.38             28.10             115.258409
      Gafsa  2023 shallow_total          33.30             54.39             163.333333
      Gafsa  2024 shallow_total          33.30             54.39             163.333333
     Kebili  2022    deep_total         236.70            539.58             227.959442
     Kebili  2022 shallow_total           5.49              0.17               3.096539
     Kebili  2023    deep_total         236.70            545.49             230.456274
     Kebili  2023 shallow_total           5.49              0.17               3.096539
     Tozeur  2021    deep_total         227.20            159.67              70.277289
     Tozeur  2021 shallow_total          48.10             45.32              94.220374
  

---
## Phase F — Master Dataset Compilation
### Cell 15: Build monthly sequence table + annual flat matrix

Two output tables:
1. **`monthly_sequences.csv`** — 4 govs × 23 years × 12 months = 1,104 rows  
   → Used for LSTM / 1D-CNN / Transformer models (Tier 2)
2. **`annual_flat_matrix.csv`** — 4 govs × 23 years = 92 rows  
   → Used for XGBoost / Ridge / Random Forest (Tier 1)  
   → Features are monthly values aggregated to annual statistics


In [ ]:
# ============================================================
# Load all source CSVs
# ============================================================

sources = {}
raw_dir = DIRS['raw_gee']

file_map = {
    'ndvi':    '01_modis_ndvi_monthly.csv',
    'lst':     '02_modis_lst_monthly.csv',
    'et':      '03_modis_et_monthly.csv',
    'era5':    '04_era5land_monthly.csv',
    'chirps':  '05_chirps_monthly.csv',
    's2':      '07_sentinel2_ndvi_ndwi_monthly.csv',
    'static':  '08_static_spatial.csv',
}

for key, fname in file_map.items():
    path = os.path.join(raw_dir, fname)
    if os.path.exists(path):
        sources[key] = pd.read_csv(path)
        print(f"✓ {key}: {sources[key].shape}")
    else:
        print(f"✗ {key}: {fname} not found — skipping")

# Load calibrated GRACE
grace_path = os.path.join(DIRS['compiled'], 'grace_calibrated_monthly.csv')
if os.path.exists(grace_path):
    sources['grace'] = pd.read_csv(grace_path)
    print(f"✓ grace (calibrated): {sources['grace'].shape}")

# Load production target
prod_path = os.path.join(DIRS['onagri'], 'production_target.csv')
df_production = pd.read_csv(prod_path)
print(f"✓ production: {df_production.shape}")

# ============================================================
# Build MONTHLY SEQUENCE TABLE
# ============================================================

keys = ['governorate', 'year', 'month']

# Start with NDVI as the base (most complete)
monthly = sources.get('ndvi', pd.DataFrame())
if monthly.empty:
    print("⚠ NDVI data missing — cannot build monthly table")
else:
    # Drop non-feature columns before merge
    drop_cols = ['image_count', 'error']

    for src_key in ['lst', 'et', 'chirps', 'grace']:
        if src_key in sources:
            df_src = sources[src_key].copy()
            feature_cols = [c for c in df_src.columns
                          if c not in keys + drop_cols
                          and 'error' not in c
                          and c != 'is_interpolated'
                          and c != 'extraction_weight']
            monthly = monthly.merge(
                df_src[keys + feature_cols].drop_duplicates(subset=keys),
                on=keys, how='left'
            )

    # Merge ERA5 (may have different structure)
    if 'era5' in sources:
        df_era5 = sources['era5']
        era5_cols = [c for c in df_era5.columns
                     if c not in keys + drop_cols and 'error' not in c]
        monthly = monthly.merge(
            df_era5[keys + era5_cols].drop_duplicates(subset=keys),
            on=keys, how='left'
        )

    # Merge Sentinel-2 (2015+ only — will be NaN for 2002-2014)
    if 's2' in sources:
        df_s2 = sources['s2']
        s2_cols = [c for c in df_s2.columns
                   if c not in keys + drop_cols and 'error' not in c
                   and c != 'S2_scene_count']
        monthly = monthly.merge(
            df_s2[keys + s2_cols].drop_duplicates(subset=keys),
            on=keys, how='left'
        )

    # Merge static features
    if 'static' in sources:
        monthly = monthly.merge(sources['static'], on='governorate', how='left')

    # Save
    path = os.path.join(DIRS['compiled'], 'monthly_sequences.csv')
    monthly.to_csv(path, index=False)
    print(f"\n✓ Monthly sequences: {monthly.shape}")
    print(f"  Columns: {list(monthly.columns)}")

# ============================================================
# Build ANNUAL FLAT MATRIX
# ============================================================

# Aggregate monthly → annual statistics
numeric_cols = [c for c in monthly.columns
                if c not in keys + ['governorate', 'year', 'month']
                and monthly[c].dtype in ['float64', 'int64', 'float32']]

# Exclude static features from aggregation
static_cols = ['elevation_mean_m', 'elevation_std_m', 'elevation_min_m',
               'elevation_max_m', 'clay_pct', 'sand_pct', 'soc_g_kg']
agg_cols = [c for c in numeric_cols if c not in static_cols]

# Annual aggregation: mean, max, min, std for each feature
annual_agg = (
    monthly
    .groupby(['governorate', 'year'])[agg_cols]
    .agg(['mean', 'max', 'min', 'std'])
    .reset_index()
)
# Flatten multi-level columns
annual_agg.columns = ['_'.join(col).rstrip('_') if col[1] else col[0]
                       for col in annual_agg.columns]

# Add critical phenological window features (not just annual averages)
# Pollination window: March-April NDVI
# Fruit development: June-September NDVI, LST
if 'NDVI_mean' in monthly.columns:
    for window_name, month_range in [('pollination', [3, 4]),
                                       ('fruit_dev', [6, 7, 8, 9]),
                                       ('harvest', [10, 11])]:
        window = monthly[monthly['month'].isin(month_range)]
        window_agg = (
            window.groupby(['governorate', 'year'])
            .agg(
                **{f'NDVI_{window_name}_mean': ('NDVI_mean', 'mean'),
                   f'NDVI_{window_name}_max': ('NDVI_mean', 'max')}
            )
            .reset_index()
        )
        annual_agg = annual_agg.merge(window_agg, on=['governorate', 'year'], how='left')

# Add lag features (previous year production)
df_prod_lag = df_production.copy()
df_prod_lag['year'] = df_prod_lag['year'] + 1
df_prod_lag = df_prod_lag.rename(columns={'production_tonnes': 'production_t_minus_1'})
annual_agg = annual_agg.merge(
    df_prod_lag[['year', 'governorate', 'production_t_minus_1']],
    on=['governorate', 'year'], how='left'
)

# Add static features
if 'static' in sources:
    annual_agg = annual_agg.merge(sources['static'], on='governorate', how='left')

# Add target variable
annual_agg = annual_agg.merge(df_production, on=['governorate', 'year'], how='left')

# Save
path = os.path.join(DIRS['compiled'], 'annual_flat_matrix.csv')
annual_agg.to_csv(path, index=False)
print(f"\n✓ Annual flat matrix: {annual_agg.shape}")
print(f"  Feature columns: {len([c for c in annual_agg.columns if c not in ['governorate','year','production_tonnes']])}")
print(f"  Target: production_tonnes")


✓ ndvi: (1104, 7)
✓ lst: (1104, 10)
✓ et: (1104, 10)
✓ era5: (1104, 22)
✓ chirps: (1104, 6)
✓ s2: (480, 8)
✓ static: (4, 8)
✓ grace (calibrated): (1104, 8)
✓ production: (92, 3)

✓ Monthly sequences: (1104, 54)
  Columns: ['governorate', 'year', 'month', 'NDVI_mean', 'NDVI_max', 'NDVI_std', 'image_count', 'LST_Day_C_mean', 'LST_Day_C_max', 'LST_Day_C_std', 'LST_Night_C_mean', 'LST_Night_C_max', 'LST_Night_C_std', 'ETa_mean', 'ETa_max', 'ETa_std', 'PET_mean', 'PET_max', 'PET_std', 'precip_total_mm', 'precip_max_daily_mm', 'rain_days', 'tws_anomaly_cm', 'tws_weighted_cm', 'tws_estimated_Mm3', 'temp_2m_K_mean', 'temp_2m_K_max', 'temp_2m_K_std', 'precip_m_mean', 'precip_m_max', 'precip_m_std', 'dewpoint_2m_K_mean', 'dewpoint_2m_K_max', 'dewpoint_2m_K_std', 'solar_rad_Jm2_mean', 'solar_rad_Jm2_max', 'solar_rad_Jm2_std', 'wind_u_ms_mean', 'wind_u_ms_max', 'wind_u_ms_std', 'wind_v_ms_mean', 'wind_v_ms_max', 'wind_v_ms_std', 'S2_NDVI_mean', 'S2_NDWI_mean', 'S2_NDVI_std', 'S2_NDWI_std', 'elevat

### Cell 16: Final inventory & dataset summary


In [ ]:
print("=" * 70)
print("FINAL DATASET COMPILATION — SUMMARY")
print("=" * 70)

for folder_key, folder_path in DIRS.items():
    if os.path.exists(folder_path):
        csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') or f.endswith('.json')]
        print(f"\n📁 {folder_key}/ ({len(csv_files)} files)")
        for f in sorted(csv_files):
            fpath = os.path.join(folder_path, f)
            size_kb = os.path.getsize(fpath) / 1024
            if f.endswith('.csv'):
                df = pd.read_csv(fpath)
                print(f"  {f:<50} {df.shape[0]:>6} rows × {df.shape[1]:>3} cols  ({size_kb:.0f} KB)")
            else:
                print(f"  {f:<50} ({size_kb:.1f} KB)")

# Key compiled files
print("\n" + "=" * 70)
print("KEY OUTPUT FILES (in 03_compiled/)")
print("=" * 70)

for fname, desc in [
    ('monthly_sequences.csv',       'Monthly table for sequence models (LSTM, CNN)'),
    ('annual_flat_matrix.csv',      'Annual table for flat models (XGBoost, Ridge)'),
    ('grace_calibrated_monthly.csv','GRACE TWS with governorate-specific calibration'),
]:
    path = os.path.join(DIRS['compiled'], fname)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"\n✅ {fname}")
        print(f"   {desc}")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        if 'governorate' in df.columns:
            print(f"   Govs: {sorted(df['governorate'].unique())}")
        if 'year' in df.columns:
            print(f"   Years: {df['year'].min()}–{df['year'].max()}")

print("\n" + "=" * 70)
print("✅ Pipeline complete. All data saved to Google Drive.")
print(f"   Location: {BASE_DIR}")
print("\n   Next step: Feature engineering notebook")
print("=" * 70)


FINAL DATASET COMPILATION — SUMMARY

📁 raw_gee/ (8 files)
  01_modis_ndvi_monthly.csv                            1104 rows ×   7 cols  (81 KB)
  02_modis_lst_monthly.csv                             1104 rows ×  10 cols  (137 KB)
  03_modis_et_monthly.csv                              1104 rows ×  10 cols  (43 KB)
  04_era5land_monthly.csv                              1104 rows ×  22 cols  (279 KB)
  05_chirps_monthly.csv                                1104 rows ×   6 cols  (73 KB)
  06_grace_tws_monthly.csv                             1104 rows ×   8 cols  (105 KB)
  07_sentinel2_ndvi_ndwi_monthly.csv                    480 rows ×   8 cols  (43 KB)
  08_static_spatial.csv                                   4 rows ×   8 cols  (1 KB)

📁 onagri/ (3 files)
  production_target.csv                                  92 rows ×   3 cols  (2 KB)
  water_calibration_full.csv                            131 rows ×   9 cols  (10 KB)
  water_calibration_totals.csv                           16 rows ×   8

In [ ]:
# ============================================================
# CELL 17: DATA QUALITY DIAGNOSTIC — Monthly Sequences
# ============================================================

import pandas as pd
import numpy as np

monthly = pd.read_csv('/content/drive/MyDrive/DatePalm_Project/03_compiled/monthly_sequences.csv')
annual = pd.read_csv('/content/drive/MyDrive/DatePalm_Project/03_compiled/annual_flat_matrix.csv')
grace = pd.read_csv('/content/drive/MyDrive/DatePalm_Project/03_compiled/grace_calibrated_monthly.csv')

print("=" * 70)
print("DIAGNOSTIC 1: MONTHLY SEQUENCES — Completeness")
print("=" * 70)

# Expected: 1104 rows (4 govs × 23 years × 12 months)
print(f"\nShape: {monthly.shape}")
print(f"Expected rows: {4 * 23 * 12} | Actual: {len(monthly)}")

# Completeness per column
total = len(monthly)
completeness = (
    monthly.drop(columns=['governorate', 'year', 'month'], errors='ignore')
    .apply(lambda col: pd.Series({
        'non_null': col.notna().sum(),
        'pct_complete': col.notna().mean() * 100,
        'all_zero': (col == 0).sum() if col.dtype in ['float64','int64'] else 0,
    }))
    .T.sort_values('pct_complete')
)

print(f"\n--- Column completeness (sorted worst → best) ---")
print(f"{'Column':<35} {'Non-null':>8} {'% Complete':>10} {'All-zero':>10}")
print("-" * 70)
for col, row in completeness.iterrows():
    flag = ""
    if row['pct_complete'] < 50:
        flag = " ⛔ CRITICAL"
    elif row['pct_complete'] < 80:
        flag = " ⚠️ WARNING"
    elif row['pct_complete'] < 95:
        flag = " 🔶 GAPS"
    print(f"{col:<35} {int(row['non_null']):>6}/{total}  {row['pct_complete']:>8.1f}%  "
          f"{int(row['all_zero']):>8}{flag}")

# Completeness by governorate
print(f"\n--- Completeness by governorate ---")
for gov in sorted(monthly['governorate'].unique()):
    sub = monthly[monthly['governorate'] == gov]
    feature_cols = [c for c in monthly.columns if c not in ['governorate','year','month']]
    null_pct = sub[feature_cols].isna().mean().mean() * 100
    print(f"  {gov}: {100-null_pct:.1f}% complete ({sub.shape[0]} rows)")

# Completeness by year (to detect temporal gaps)
print(f"\n--- Completeness by year (average across all feature columns) ---")
feature_cols = [c for c in monthly.columns if c not in ['governorate','year','month']]
yearly_complete = monthly.groupby('year')[feature_cols].apply(
    lambda x: x.notna().mean().mean() * 100
)
for year, pct in yearly_complete.items():
    bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
    flag = " ⚠️" if pct < 80 else ""
    print(f"  {year}: {bar} {pct:.1f}%{flag}")

DIAGNOSTIC 1: MONTHLY SEQUENCES — Completeness

Shape: (1104, 54)
Expected rows: 1104 | Actual: 1104

--- Column completeness (sorted worst → best) ---
Column                              Non-null % Complete   All-zero
----------------------------------------------------------------------
ETa_std                                192/1104      17.4%         0 ⛔ CRITICAL
PET_mean                               192/1104      17.4%         0 ⛔ CRITICAL
PET_std                                192/1104      17.4%         0 ⛔ CRITICAL
PET_max                                192/1104      17.4%         0 ⛔ CRITICAL
ETa_mean                               192/1104      17.4%         0 ⛔ CRITICAL
ETa_max                                192/1104      17.4%         0 ⛔ CRITICAL
S2_NDVI_std                            447/1104      40.5%         0 ⛔ CRITICAL
S2_NDWI_std                            447/1104      40.5%         0 ⛔ CRITICAL
S2_NDVI_mean                           447/1104      40.5%         0 ⛔

In [ ]:
# ============================================================
# CELL 18: DATA QUALITY DIAGNOSTIC — Annual Flat Matrix
# ============================================================

print("=" * 70)
print("DIAGNOSTIC 2: ANNUAL FLAT MATRIX — Completeness & Distributions")
print("=" * 70)

print(f"\nShape: {annual.shape}")
print(f"Expected rows: 92 | Actual: {len(annual)}")
print(f"Total features: {len(annual.columns) - 3} (excl. governorate, year, production_tonnes)")

# Classify columns by completeness
full_cols = []
partial_cols = []
empty_cols = []
near_empty_cols = []

for col in annual.columns:
    if col in ['governorate', 'year']:
        continue
    n = annual[col].notna().sum()
    pct = n / len(annual) * 100
    if pct >= 95:
        full_cols.append((col, pct))
    elif pct >= 50:
        partial_cols.append((col, pct))
    elif pct > 0:
        near_empty_cols.append((col, pct))
    else:
        empty_cols.append((col, pct))

print(f"\n--- Column classification ---")
print(f"  ✅ Full (≥95%):        {len(full_cols)} columns")
print(f"  🔶 Partial (50–95%):   {len(partial_cols)} columns")
print(f"  ⛔ Near-empty (<50%):  {len(near_empty_cols)} columns")
print(f"  💀 Completely empty:    {len(empty_cols)} columns")

if near_empty_cols:
    print(f"\n--- ⛔ Near-empty columns (< 50% complete) ---")
    print(f"  These should be DROPPED or investigated before modeling.\n")
    for col, pct in sorted(near_empty_cols, key=lambda x: x[1]):
        n_valid = annual[col].notna().sum()
        print(f"  {col:<45} {n_valid:>3}/{len(annual)} rows ({pct:.1f}%)")

if empty_cols:
    print(f"\n--- 💀 Completely empty columns ---")
    for col, pct in empty_cols:
        print(f"  {col}")

# Specifically investigate the ET/PET issue
print(f"\n--- ETa/PET deep dive ---")
et_cols = [c for c in annual.columns if 'ETa' in c or 'PET' in c]
if et_cols:
    et_sub = annual[['governorate', 'year'] + et_cols]
    print(f"  ET/PET columns: {len(et_cols)}")
    print(f"  Rows with ANY ET data: {et_sub[et_cols].notna().any(axis=1).sum()}/{len(annual)}")
    has_data = et_sub[et_sub[et_cols[0]].notna()]
    if len(has_data) > 0:
        print(f"\n  Rows with ET data:")
        print(has_data[['governorate', 'year'] + et_cols[:4]].to_string(index=False))
    print(f"\n  ⚠️ DIAGNOSIS: MODIS MOD16A2 masks arid regions as 'no data'.")
    print(f"  The algorithm requires minimum vegetation cover (NDVI > ~0.1).")
    print(f"  In southern Tunisia, only dense oasis patches meet this threshold.")
    print(f"  RECOMMENDATION: Drop all ET/PET columns from the model.")
    print(f"  Alternative: Use ERA5-Land potential evaporation instead.")

# Target variable check
print(f"\n--- Target variable: production_tonnes ---")
if 'production_tonnes' in annual.columns:
    prod = annual[['governorate', 'year', 'production_tonnes']].dropna()
    print(f"  Non-null: {len(prod)}/{len(annual)}")
    print(f"\n  Per governorate:")
    for gov in sorted(annual['governorate'].unique()):
        sub = annual[annual['governorate'] == gov]['production_tonnes']
        print(f"    {gov}: mean={sub.mean():,.0f}t  std={sub.std():,.0f}t  "
              f"min={sub.min():,.0f}t  max={sub.max():,.0f}t  range_ratio={sub.max()/sub.min():.1f}x")

# Lag feature check
print(f"\n--- Lag feature: production_t_minus_1 ---")
if 'production_t_minus_1' in annual.columns:
    lag = annual['production_t_minus_1']
    print(f"  Non-null: {lag.notna().sum()}/{len(annual)}")
    print(f"  Missing in first year (expected): {lag.isna().sum()}")

DIAGNOSTIC 2: ANNUAL FLAT MATRIX — Completeness & Distributions

Shape: (92, 193)
Expected rows: 92 | Actual: 92
Total features: 190 (excl. governorate, year, production_tonnes)

--- Column classification ---
  ✅ Full (≥95%):        151 columns
  🔶 Partial (50–95%):   0 columns
  ⛔ Near-empty (<50%):  40 columns
  💀 Completely empty:    0 columns

--- ⛔ Near-empty columns (< 50% complete) ---
  These should be DROPPED or investigated before modeling.

  ETa_mean_mean                                  16/92 rows (17.4%)
  ETa_mean_max                                   16/92 rows (17.4%)
  ETa_mean_min                                   16/92 rows (17.4%)
  ETa_mean_std                                   16/92 rows (17.4%)
  ETa_max_mean                                   16/92 rows (17.4%)
  ETa_max_max                                    16/92 rows (17.4%)
  ETa_max_min                                    16/92 rows (17.4%)
  ETa_max_std                                    16/92 rows (17.4%)


In [ ]:
# ============================================================
# CELL 19: DATA QUALITY DIAGNOSTIC — GRACE Calibrated
# ============================================================

print("=" * 70)
print("DIAGNOSTIC 3: GRACE CALIBRATED TWS")
print("=" * 70)

print(f"\nShape: {grace.shape}")
print(f"Expected: {4 * 23 * 12} = 1104 | Actual: {len(grace)}")

if len(grace) > 1104:
    print(f"  ⚠️ {len(grace) - 1104} extra rows — likely duplicate months in GRACE source")
    dupes = grace.duplicated(subset=['governorate', 'year', 'month'], keep=False)
    if dupes.any():
        print(f"  Duplicate (gov, year, month) entries: {dupes.sum()}")
        print(f"\n  Sample duplicates:")
        print(grace[dupes].sort_values(['governorate','year','month']).head(10).to_string(index=False))
        print(f"\n  FIX: Keep first occurrence per (gov, year, month)")

# TWS statistics
print(f"\n--- TWS anomaly statistics ---")
print(f"  Range: {grace['tws_anomaly_cm'].min():.2f} to {grace['tws_anomaly_cm'].max():.2f} cm")
print(f"  Mean:  {grace['tws_anomaly_cm'].mean():.2f} cm")
print(f"  Trend: ", end="")
early = grace[(grace['year'] <= 2005)]['tws_anomaly_cm'].mean()
late = grace[(grace['year'] >= 2020)]['tws_anomaly_cm'].mean()
print(f"{early:.2f} cm (2002-05 avg) → {late:.2f} cm (2020-24 avg) = {late-early:+.2f} cm change")
if late < early:
    print(f"  ✓ Negative trend confirms groundwater depletion signal")

# Interpolation check
if 'is_interpolated' in grace.columns:
    n_interp = grace['is_interpolated'].sum()
    print(f"\n--- Interpolated values ---")
    print(f"  Interpolated months: {n_interp}/{len(grace)} ({n_interp/len(grace)*100:.1f}%)")
    interp_months = grace[grace['is_interpolated']==True][['year','month']].drop_duplicates()
    if len(interp_months) > 0:
        print(f"  Gap period:")
        print(interp_months.to_string(index=False))

# Calibration weight check
if 'extraction_weight' in grace.columns:
    print(f"\n--- Extraction weights ---")
    weights = grace.groupby('governorate')['extraction_weight'].first()
    for gov, w in weights.items():
        bar = "█" * int(w * 50)
        print(f"  {gov:<10}: {w:.3f} {bar}")

DIAGNOSTIC 3: GRACE CALIBRATED TWS

Shape: (1104, 8)
Expected: 1104 = 1104 | Actual: 1104

--- TWS anomaly statistics ---
  Range: -18.71 to 2.46 cm
  Mean:  -6.89 cm
  Trend: 0.40 cm (2002-05 avg) → -16.26 cm (2020-24 avg) = -16.66 cm change
  ✓ Negative trend confirms groundwater depletion signal

--- Interpolated values ---
  Interpolated months: 152/1104 (13.8%)
  Gap period:
 year  month
 2002      1
 2002      2
 2002      3
 2002      6
 2002      7
 2003      6
 2011      1
 2011      6
 2011     12
 2012      5
 2012     10
 2013      3
 2013      8
 2013      9
 2014      2
 2014      7
 2014     12
 2015      5
 2015      6
 2015     10
 2015     11
 2016      4
 2016      9
 2016     10
 2017      2
 2017      7
 2017      8
 2017      9
 2017     10
 2017     11
 2017     12
 2018      1
 2018      2
 2018      3
 2018      4
 2018      5
 2018      8
 2018      9

--- Extraction weights ---
  Gabes     : 0.169 ████████
  Gafsa     : 0.056 ██
  Kebili    : 0.562 ██████████

In [ ]:
# ============================================================
# CELL 20: RECOMMENDATIONS — Pre-modeling cleanup actions
# ============================================================

print("=" * 70)
print("PRE-MODELING CLEANUP RECOMMENDATIONS")
print("=" * 70)

# Identify columns to drop
drop_candidates = []
for col in annual.columns:
    if col in ['governorate', 'year', 'production_tonnes']:
        continue
    pct = annual[col].notna().mean() * 100
    if pct < 50:
        drop_candidates.append(col)

print(f"""
1. DROP NEAR-EMPTY COLUMNS ({len(drop_candidates)} columns)
   All ET/PET columns from MODIS MOD16A2 — insufficient coverage
   in arid southern Tunisia. Sentinel-2 columns before 2015 are
   expected NaN and handled separately.

2. FIX GRACE DUPLICATES
   Deduplicate on (governorate, year, month) keeping first value.

3. VERIFY ERA5-LAND COLUMNS
   If student already has ERA5-Land extracted independently,
   compare values for consistency before deciding which to keep.

4. FEATURE COUNT AFTER CLEANUP
   Current: {len(annual.columns) - 3} features
   After dropping near-empty: ~{len(annual.columns) - 3 - len(drop_candidates)} features
   This is a healthier ratio for 92 samples.

5. NEXT STEPS FOR FEATURE ENGINEERING
   - Compute GDD (base 18°C) from LST
   - Compute VPD from ERA5 temperature + dewpoint
   - Compute SPI-3, SPI-6 drought indices from CHIRPS
   - Correlation filtering to reduce redundant features
   - Train/test split: 2002–2020 / 2021–2024
""")

# Auto-generate the clean column list
usable_cols = [c for c in annual.columns if c not in drop_candidates]
print(f"Usable columns ({len(usable_cols)}):")
for i, col in enumerate(usable_cols):
    if col not in ['governorate', 'year', 'production_tonnes']:
        pct = annual[col].notna().mean() * 100
        print(f"  {i:3d}. {col:<45} {pct:.0f}%")

PRE-MODELING CLEANUP RECOMMENDATIONS

1. DROP NEAR-EMPTY COLUMNS (40 columns)
   All ET/PET columns from MODIS MOD16A2 — insufficient coverage
   in arid southern Tunisia. Sentinel-2 columns before 2015 are
   expected NaN and handled separately.

2. FIX GRACE DUPLICATES
   Deduplicate on (governorate, year, month) keeping first value.

3. VERIFY ERA5-LAND COLUMNS
   If student already has ERA5-Land extracted independently,
   compare values for consistency before deciding which to keep.

4. FEATURE COUNT AFTER CLEANUP
   Current: 190 features
   After dropping near-empty: ~150 features
   This is a healthier ratio for 92 samples.

5. NEXT STEPS FOR FEATURE ENGINEERING
   - Compute GDD (base 18°C) from LST
   - Compute VPD from ERA5 temperature + dewpoint
   - Compute SPI-3, SPI-6 drought indices from CHIRPS
   - Correlation filtering to reduce redundant features
   - Train/test split: 2002–2020 / 2021–2024

Usable columns (153):
    2. NDVI_mean_mean                                100